In [1]:
# %%

# Phillies Applied Biomechanics QA -- Velocity Model & FCL-2025 Ranking

**Question:** of the pitchers in the 2025 FCL dataset, who will throw hardest in five years?

While future velocity measurements were not available, the objective of this analysis was to identify the pitcher most likely to throw the hardest five years from now. To address this, I developed a projection framework that combines current biomechanical performance, kinetic-chain efficiency, movement archetypes, and developmental indicators into a future velocity potential estimate.

This is fundamentally a question about *biomechanics telling a story about velocity*. The outcome shows a distribuiton of the predicted maximum values base factors stated above rather than a point estamte as "hardest in
five years" as a question about an uncertain future maximum, not a single point estimate. To validate the signal of the rankings against iteslt, several independent methods were implemeneted before trusting it. The approach here: understand what mechanical dynamics
actually relate to velocity and why; discover natural mechanical groupings (archetypes) among
pitchers; build a disciplined, validated model on top of that understanding; Rank picthers from a concensus of combine methods and statistical validation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, gumbel_r, kstest

from sklearn.model_selection import GroupKFold, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, silhouette_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import umap
from xgboost import XGBRegressor
import shap

RANDOM_STATE = 42
PROJECT_DIR = r'C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026'
FIG_DIR = os.path.join(PROJECT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid')
PHI_RED = 'crimson'
PHI_BLUE = 'navy'
PHI_PURPLE = 'rebeccapurple'
PALETTE = sns.color_palette([PHI_RED, PHI_BLUE, PHI_PURPLE, 'darkseagreen', 'goldenrod', 'dimgray', 'teal'])

def safe_to_csv(dataframe, path, **kwargs):
    try:
        dataframe.to_csv(path, **kwargs)
        print('saved', path, flush=True)
        return path
    except PermissionError:
        fallback = path.replace('.csv', '_OUTPUT.csv')
        dataframe.to_csv(fallback, **kwargs)
        print(f'WARNING: {path} is locked by another process (likely open in an editor/preview tab) -- '
              f'wrote to {fallback} instead. Close the locked file and rename.', flush=True)
        return fallback

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print('saved', path, flush=True)

metric_desc = pd.read_csv(os.path.join(PROJECT_DIR, 'AB_QA_Questionnaire_2026', 'metric_descriptions.csv'))
metric_desc_map = dict(zip(metric_desc['Column'], metric_desc['Description']))
metric_units_map = dict(zip(metric_desc['Column'], metric_desc['Units'].fillna('')))

## 1. The Data

In [3]:
LABELED_PATH = os.path.join(PROJECT_DIR, 'AB_QA_Questionnaire_2026', 'velo_and_mechanics.csv')
FCL_PATH = os.path.join(PROJECT_DIR, 'AB_QA_Questionnaire_2026', 'fcl_mechanics_2025.csv')
TARGET = 'velocity'
PITCHER_COL = 'pitcher_id'
DATE_COL = 'date'

df = pd.read_csv(LABELED_PATH)
fcl = pd.read_csv(FCL_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
fcl[DATE_COL] = pd.to_datetime(fcl[DATE_COL])
df['orig_order'] = np.arange(len(df))
fcl['orig_order'] = np.arange(len(fcl))
df['season'] = df[DATE_COL].dt.year
df['level'] = np.where(df['season'] == 2024, 'FCL_2024', 'FSL_2025')

print('Labeled (velocity known):', df.shape, ' FCL-2025 target (velocity unknown):', fcl.shape)
print('Pitchers labeled:', df[PITCHER_COL].nunique(), ' Pitchers in FCL-2025 target:', fcl[PITCHER_COL].nunique())
print('Overlap pitchers:', len(set(df[PITCHER_COL]) & set(fcl[PITCHER_COL])))

raw_mechanics_cols = [c for c in df.columns if c in fcl.columns
                      and c not in [DATE_COL, PITCHER_COL, 'pitcher_handedness', 'orig_order']
                      and df[c].dtype != object]
print('n raw mechanics columns shared by both files:', len(raw_mechanics_cols))

Labeled (velocity known): (8981, 70)  FCL-2025 target (velocity unknown): (6893, 67)
Pitchers labeled: 203  Pitchers in FCL-2025 target: 71
Overlap pitchers: 0
n raw mechanics columns shared by both files: 63


**Zero pitchers overlap between the labeled file and the FCL-2025 target file** -- the 203
pitchers with known velocity and the 71 FCL-2025 pitchers being ranked are two entirely different
groups of people, so there's no shortcut of looking up a target pitcher's own historical velocity.
Every FCL-2025 pitcher has to be evaluated purely from his mechanics, by way of a relationship
learned on the labeled cohort. (Separately, *within* the labeled file itself, a small number of
pitchers were tracked in both 2024 and 2025 -- that's a different comparison, covered next.)

## 2. What Do the Mechanics Actually Say About Velocity? (Exploratory Analysis)

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x='level', y=TARGET, ax=axes[0], palette=[PHI_BLUE, PHI_RED])
axes[0].set_title('Velocity by Level (2024 Rookie-Ball vs 2025 Single-A)')
axes[0].set_xlabel('')
axes[0].set_ylabel('Velocity (mph)')
monthly = df.groupby(df[DATE_COL].dt.to_period('M'))[TARGET].mean()
for lvl, g in df.groupby('level'):
    m = g.groupby(g[DATE_COL].dt.to_period('M'))[TARGET].mean()
    axes[1].plot(m.index.astype(str), m.values, marker='o', label=lvl)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].legend()
axes[1].set_title('Mean Velocity by Month, Within Level')
axes[1].set_ylabel('Velocity (mph)')
savefig('01_level_confound.png')

print(df.groupby('level')[TARGET].describe())

season_counts = df.groupby(PITCHER_COL)['season'].nunique()
dev_pitchers_idx = season_counts[season_counts > 1].index
dev_compare = (
    df[df[PITCHER_COL].isin(dev_pitchers_idx)]
      .groupby([PITCHER_COL, 'season'])[TARGET].mean().unstack()
)
dev_compare['delta'] = dev_compare[2025] - dev_compare[2024]
n_dev_pitchers = len(dev_compare)
global_dev_delta = dev_compare['delta'].mean()
print(f'{n_dev_pitchers} pitchers appear in both 2024 (rookie ball) and 2025 (Single-A); '
      f'mean within-pitcher gain = {global_dev_delta:.2f} mph')

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\01_level_confound.png
           count       mean       std        min        25%        50%  \
level                                                                    
FCL_2024  1609.0  91.544417  2.740383  68.955808  89.953131  91.325712   
FSL_2025  7372.0  92.652747  2.624501  81.816432  91.004805  92.689169   

                75%         max  
level                            
FCL_2024  93.395237   98.411321  
FSL_2025  94.449777  101.334670  
10 pitchers appear in both 2024 (rookie ball) and 2025 (Single-A); mean within-pitcher gain = 0.62 mph


This comparison is entirely *within the labeled file* -- 2024 and 2025 here are two seasons of
the same 203-pitcher labeled cohort. The 10 pitchers
just identified are the ones who happen to appear in both of those seasons. Likely a promotion between seasons to the minor leagues.

Those two seasons are not just two points in time -- 2024 is rookie ball, 2025 is Single-A, a
level jump. The ~1.1 mph pooled gap between them is mostly a **promotion effect** (who got moved
up), not a clean time trend: the 10 pitchers actually seen at both levels gained only about half
that much on average, in their own paired before/after comparison. Both numbers matter for
different reasons -- the pooled gap would overstate individual development if used naively, while
the paired number is the closest thing this dataset has to direct evidence of how much a real
pitcher actually grows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df[TARGET], kde=True, ax=axes[0], color=PHI_RED)
axes[0].set_title('Pitch-Level Velocity Distribution')
axes[0].set_xlabel('Velocity (mph)')
pitcher_velo = df.groupby(PITCHER_COL)[TARGET].mean()
sns.histplot(pitcher_velo, kde=True, ax=axes[1], color=PHI_BLUE)
axes[1].set_title('Per-Pitcher Mean Velocity Distribution')
axes[1].set_xlabel('Velocity (mph)')
savefig('02_velocity_distribution.png')

numeric_cols = [c for c in raw_mechanics_cols if c not in ['season']]
pooled_corr = df[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET)
pooled_corr_sorted = pooled_corr.reindex(pooled_corr.abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 8))
top20 = pooled_corr_sorted.head(20).index.tolist()
sns.heatmap(df[top20 + [TARGET]].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Velocity and Its Top 20 Most-Correlated Biomechanics')
savefig('03_correlation_heatmap.png')

df_centered = df.copy()
df_centered[numeric_cols] = df_centered.groupby(PITCHER_COL)[numeric_cols].transform(lambda x: x - x.mean())
within_corr = df_centered[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET)

comp = pd.DataFrame({'across pitchers': pooled_corr_sorted, 'within a pitcher': within_corr}).reindex(
    pooled_corr_sorted.index).head(15)
comp.plot(kind='barh', figsize=(9, 7), color=[PHI_RED, PHI_BLUE])
plt.title('What Predicts Velocity Across Pitchers vs. Within One Pitcher\'s Own Outings')
plt.xlabel('Correlation with velocity')
plt.gca().invert_yaxis()
savefig('04_pooled_vs_within_pitcher.png')

<!-- Two different questions, two different answers. "Across pitchers," the strongest correlates are
torso posture/control and rotational speed -- this is "what kind of pitcher throws hard."
"Within one pitcher's own outings" (de-meaning by pitcher first) asks a narrower question: what
makes *this* pitcher's fastest pitches faster than his own average -- relevant context, since the
ranking task is squarely the first question, not the second. -->




These analyses answer two different questions. Looking across all pitchers identifies the biomechanical characteristics associated with high velocity overall—in other words, "what types of pitchers tend to throw hard?" In this dataset, torso posture/control and rotational speed emerged as the strongest velocity-related traits.

By contrast, de-meaning the data within each pitcher focuses on a different question: "what makes a particular pitcher's hardest pitches faster than his own average?" This isolates pitch-to-pitch variation within an individual rather than differences between pitchers.

While both perspectives provide useful insight, the ranking task is fundamentally a between-pitcher problem. The objective is not to explain why a pitcher occasionally throws harder than his personal average, but rather to identify which pitchers possess the biomechanical characteristics associated with the highest long-term velocity potential.

In [6]:
pitch_counts = df.groupby(PITCHER_COL).size().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(pitch_counts, bins=30, ax=axes[0], color=PHI_BLUE)
axes[0].set_title('Pitches Recorded per Pitcher')
axes[0].set_xlabel('pitches')
cum_share = pitch_counts.sort_values(ascending=False).cumsum() / pitch_counts.sum()
axes[1].plot(range(1, len(cum_share) + 1), cum_share.values, color=PHI_RED)
axes[1].set_xlabel('Pitcher rank, by volume')
axes[1].set_ylabel('Cumulative share of all pitches')
axes[1].set_title('How Concentrated Is the Sample?')
savefig('05_pitcher_overrepresentation.png')

top10_share = cum_share.iloc[9]
print(f'Top 10 of {len(pitch_counts)} pitchers account for {top10_share:.1%} of all recorded pitches')

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\05_pitcher_overrepresentation.png
Top 10 of 203 pitchers account for 27.3% of all recorded pitches


Pitcher Overrepresentation:
The top 10 highest-volume pitchers account for over a quarter of every pitch in the dataset.
Left uncorrected, both correlation analysis and model training would mostly be learning about a
handful of high-volume arms. **Fix applied throughout the rest of this notebook:** group pitches
by `pitcher_id` for cross-validation (no pitcher's pitches ever appear in both a training fold and
its matching validation fold) and weight each pitch by the inverse of how many pitches that
pitcher threw, so every pitcher counts equally regardless of sample size.

## 3. Engineering Features That Encode a Biomechanical Idea, Not Just a Raw Angle
**Hypothesis:** 
The central biomechanical concept underlying this analysis is the kinetic chain. Pitching velocity is not generated by any single joint, angle, or body segment acting independently. Rather, velocity emerges from the coordinated transfer of energy through a sequence of linked movements, beginning with force generation in the lower body, progressing through pelvic and trunk rotation, and ultimately transferring into the throwing arm and the baseball.

This perspective motivated the feature engineering strategy used throughout the analysis. Rather than evaluating raw biomechanical measurements in isolation, I sought to construct features that better represent how efficiently energy moves through the kinetic chain. For example, stride length was normalized relative to body size to capture mechanical leverage rather than absolute distance. More importantly, biomechanical variables were grouped into functional systems representing lower-body force production, pelvic rotation, hip-shoulder separation, trunk rotation, and arm acceleration. These systems provide a more interpretable representation of kinetic energy transfer than any individual joint angle alone.

The underlying hypothesis is that elite velocity is produced not simply by rotating one segment faster, but by efficiently sequencing and transferring energy from proximal segments to distal segments. Consequently, a pitcher with a highly efficient kinetic chain may possess greater long-term velocity potential than a pitcher who achieves similar current velocity through isolated segmental speed alone. This concept became a foundational component of both the exploratory analysis and the final projection framework.


Two ideas worth testing explicitly, beyond the raw joint angles and angular velocities already in
the data: (1) a pitcher's stride should scale with his own body, not be read as an absolute
number; (2) pitching is a **kinetic chain** -- energy is supposed to flow proximal-to-distal,
pelvis to torso to arm -- so a feature that captures *how that energy moves through the body*,
not just how fast any one segment rotates, ought to say something about velocity that no single
raw angle does on its own.

In [7]:
def compute_outing_drift(data, value_col, pitcher_col=PITCHER_COL, date_col=DATE_COL,
                          order_col='orig_order', min_pitches=3):
    drifts = {}
    sorted_data = data.sort_values(order_col)
    for (pid, date), g in sorted_data.groupby([pitcher_col, date_col]):
        if len(g) >= min_pitches:
            yv = g[value_col].to_numpy()
            if np.std(yv) > 0:
                xv = np.arange(len(yv))
                slope = np.polyfit(xv, yv, 1)[0]
                drifts.setdefault(pid, []).append(slope)
    return pd.Series({pid: float(np.mean(v)) for pid, v in drifts.items()}, name=f'{value_col}_outing_drift')

arm_slot_drift_df = compute_outing_drift(df, 'arm_slot')
stride_drift_df = compute_outing_drift(df, 'stride_length')
velo_drift_df = compute_outing_drift(df, 'velocity')
arm_slot_drift_fcl = compute_outing_drift(fcl, 'arm_slot')
stride_drift_fcl = compute_outing_drift(fcl, 'stride_length')

df['arm_slot_outing_drift'] = df[PITCHER_COL].map(arm_slot_drift_df).fillna(0)
df['stride_length_outing_drift'] = df[PITCHER_COL].map(stride_drift_df).fillna(0)
fcl['arm_slot_outing_drift'] = fcl[PITCHER_COL].map(arm_slot_drift_fcl).fillna(0)
fcl['stride_length_outing_drift'] = fcl[PITCHER_COL].map(stride_drift_fcl).fillna(0)

def engineer_mechanics_features(data):
    d = data.copy()
    eps = 1e-6
    d['stride_length_norm'] = d['stride_length'] / d['player_height']
    d['stride_angle_dev'] = d['stride_angle'].abs()
    # Allometric scaling (log-log linearizes power-law body-size relationships -- standard practice
    # for size-normalized biomechanical variables; see Section 6 markdown for the citation) gives a
    # second, literature-motivated functional form of the same normalized-stride idea.
    d['stride_length_norm_log'] = np.log(d['stride_length_norm'].clip(lower=eps))

    rot_weights = {
        'pelvis_rotation_velo_max': 0.22, 'torso_rotation_velo_max': 0.18,
        'back_leg_external_rotation_velo_max': 0.12, 'lead_leg_internal_rotation_velo_max': 0.12,
        'lead_knee_extension_velo_max': 0.10, 'throw_shoulder_internal_rotation_velo_max': 0.16,
        'throw_elbow_extension_velo_max': 0.10,
    }
    rot_energy = np.zeros(len(d))
    for col, w in rot_weights.items():
        omega_rad = np.deg2rad(d[col])
        rot_energy += w * (omega_rad ** 2)
    d['rotational_ke_proxy'] = rot_energy
    d['linear_ke_proxy'] = (d['center_of_mass_velo_max'] ** 2) * d['player_height']

    # Raw-ratio sequencing proxy (magnitude only, clipped to bound outliers from near-zero
    # denominators) ...
    seq1 = (d['torso_rotation_velo_max'].abs() / (d['pelvis_rotation_velo_max'].abs() + eps)).clip(0, 5)
    seq2 = (d['throw_shoulder_internal_rotation_velo_max'].abs() / (d['torso_rotation_velo_max'].abs() + eps)).clip(0, 5)
    d['seq_ratio_pelvis_to_torso'] = seq1
    d['seq_ratio_torso_to_arm'] = seq2
    d['kinetic_chain_efficiency'] = d[['seq_ratio_pelvis_to_torso', 'seq_ratio_torso_to_arm']].mean(axis=1)
    # ... and a log-ratio version of the same idea. A log-ratio is the standard form for comparing
    # two positive, multiplicatively-related quantities (symmetric around zero, not bounded below by
    # zero the way a raw ratio is, and not dependent on an arbitrary clip) -- both forms are kept as
    # separate candidates below and stability selection (Section 6) decides empirically which
    # functional form, if either, actually carries signal.
    d['seq_logratio_pelvis_to_torso'] = np.log((d['torso_rotation_velo_max'].abs() + eps) /
                                                (d['pelvis_rotation_velo_max'].abs() + eps))
    d['seq_logratio_torso_to_arm'] = np.log((d['throw_shoulder_internal_rotation_velo_max'].abs() + eps) /
                                             (d['torso_rotation_velo_max'].abs() + eps))
    return d

df = engineer_mechanics_features(df)
fcl = engineer_mechanics_features(fcl)

engineered_extra_cols = [
    'stride_length_norm', 'stride_length_norm_log', 'stride_angle_dev', 'rotational_ke_proxy',
    'linear_ke_proxy', 'seq_ratio_pelvis_to_torso', 'seq_ratio_torso_to_arm',
    'seq_logratio_pelvis_to_torso', 'seq_logratio_torso_to_arm', 'kinetic_chain_efficiency',
    'arm_slot_outing_drift', 'stride_length_outing_drift',
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=df, x='stride_length_norm', y=TARGET, ax=axes[0],
            scatter_kws={'alpha': 0.15, 'color': PHI_BLUE}, line_kws={'color': PHI_RED})
axes[0].set_title('Stride Length (% of height) vs Velocity')
sns.regplot(data=df, x='kinetic_chain_efficiency', y=TARGET, ax=axes[1],
            scatter_kws={'alpha': 0.15, 'color': PHI_BLUE}, line_kws={'color': PHI_RED})
axes[1].set_title('Kinetic-Chain Sequencing Efficiency vs Velocity')
savefig('06_stride_efficiency.png')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=df, x='rotational_ke_proxy', y=TARGET, ax=axes[0],
            scatter_kws={'alpha': 0.15, 'color': PHI_BLUE}, line_kws={'color': PHI_RED})
axes[0].set_title('Rotational Kinetic-Energy Proxy vs Velocity')
sns.regplot(data=df, x='torso_rotation_velo_max', y=TARGET, ax=axes[1],
            scatter_kws={'alpha': 0.15, 'color': PHI_BLUE}, line_kws={'color': PHI_RED})
axes[1].set_title('Raw Torso Rotation Velocity vs Velocity')
savefig('07_kinetic_chain.png')

print('corr stride_length_norm:', round(df['stride_length_norm'].corr(df[TARGET]), 3))
print('corr kinetic_chain_efficiency:', round(df['kinetic_chain_efficiency'].corr(df[TARGET]), 3))
print('corr rotational_ke_proxy:', round(df['rotational_ke_proxy'].corr(df[TARGET]), 3))
print('corr torso_rotation_velo_max (raw ingredient):', round(df['torso_rotation_velo_max'].corr(df[TARGET]), 3))

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\06_stride_efficiency.png
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\07_kinetic_chain.png
corr stride_length_norm: 0.085
corr kinetic_chain_efficiency: 0.017
corr rotational_ke_proxy: 0.13
corr torso_rotation_velo_max (raw ingredient): 0.288


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(velo_drift_df, bins=25, ax=axes[0], color=PHI_RED)
axes[0].axvline(0, color='black', linestyle='--')
axes[0].set_title('Within-Outing Velocity Drift (mph per pitch)')
axes[1].scatter(pitch_counts.reindex(velo_drift_df.index), velo_drift_df, alpha=0.5, color=PHI_BLUE)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Total pitches thrown (season volume)')
axes[1].set_ylabel('Velocity drift (mph per pitch, within an outing)')
axes[1].set_title('Workload vs In-Outing Fatigue')
savefig('08_fatigue_drift.png')
print('Mean within-outing velocity drift:', round(velo_drift_df.mean(), 4), 'mph/pitch (essentially flat)')

plt.figure(figsize=(9, 6))
dev_plot = dev_compare.sort_values('delta', ascending=False)
colors = [PHI_RED if v < 0 else PHI_BLUE for v in dev_plot['delta']]
plt.bar(range(len(dev_plot)), dev_plot['delta'], color=colors)
plt.axhline(global_dev_delta, color='black', linestyle='--', label=f'mean = {global_dev_delta:.2f} mph')
plt.xlabel('Pitcher (sorted by year-over-year change)')
plt.ylabel('2025 minus 2024 mean velocity (mph)')
plt.title('Year-over-Year Development: the Few Pitchers Seen at Both Levels')
plt.legend()
savefig('09_dual_year_development.png')
print('done with exploratory section')

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\08_fatigue_drift.png
Mean within-outing velocity drift: -0.0075 mph/pitch (essentially flat)
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\09_dual_year_development.png
done with exploratory section


## 4. Discovering Mechanical Archetypes

Rather than assume every pitcher's mechanics matter in the same way, this reduces every pitch's
full mechanics profile to a 2D embedding (UMAP) and clusters it (KMeans) -- fit on the **pooled**
labeled + FCL-2025 data, using no velocity information at all, so the two cohorts land in a shared
space and an FCL-2025 pitcher can be meaningfully compared to the labeled population. This finds
natural mechanical "types" of pitcher, independent of outcome, which becomes useful later both as
a model feature and as a way to describe *why* a given pitcher resembles -- or doesn't resemble --
pitchers who throw hard.

In [9]:
archetype_feature_cols = ['stride_length_norm', 'stride_angle_dev', 'arm_slot_outing_drift', 'stride_length_outing_drift']
shared_feature_cols = [c for c in raw_mechanics_cols if c != 'season'] + archetype_feature_cols
shared_feature_cols = [c for c in shared_feature_cols if c in df.columns and c in fcl.columns]
print('n shared feature cols for UMAP:', len(shared_feature_cols))

df_pool = df[shared_feature_cols].copy()
df_pool['__source__'] = 'labeled'
fcl_pool = fcl[shared_feature_cols].copy()
fcl_pool['__source__'] = 'FCL-2025 target'
pooled = pd.concat([df_pool, fcl_pool], ignore_index=True)

imputer_pool = SimpleImputer(strategy='median')
scaler_pool = StandardScaler()
X_pool_scaled = scaler_pool.fit_transform(imputer_pool.fit_transform(pooled[shared_feature_cols]))

reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=20, min_dist=0.3)
embedding = reducer.fit_transform(X_pool_scaled)
pooled['umap_1'] = embedding[:, 0]
pooled['umap_2'] = embedding[:, 1]

best_k, best_score, best_labels = None, -1, None
sil_scores = {}
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(embedding)
    score = silhouette_score(embedding, labels)
    sil_scores[k] = score
    if score > best_score:
        best_k, best_score, best_labels = k, score, labels
pooled['mech_archetype'] = best_labels.astype(str)
print(f'Silhouette by k: {sil_scores}')
print(f'Chosen k={best_k} clusters (silhouette={best_score:.3f})')

n_train = len(df)
df['umap_1'] = pooled['umap_1'].values[:n_train]
df['umap_2'] = pooled['umap_2'].values[:n_train]
df['mech_archetype'] = pooled['mech_archetype'].values[:n_train]
fcl['umap_1'] = pooled['umap_1'].values[n_train:]
fcl['umap_2'] = pooled['umap_2'].values[n_train:]
fcl['mech_archetype'] = pooled['mech_archetype'].values[n_train:]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.scatterplot(data=pooled, x='umap_1', y='umap_2', hue='mech_archetype', palette=PALETTE,
                 alpha=0.6, s=20, ax=axes[0], legend='full')
axes[0].set_title(f'Mechanical Archetypes (k={best_k} clusters)')
sns.scatterplot(data=pooled, x='umap_1', y='umap_2', hue='__source__',
                 palette=[PHI_BLUE, PHI_RED], alpha=0.5, s=20, ax=axes[1])
axes[1].set_title('Labeled Pitches vs FCL-2025 Target Pitches (Shared Embedding)')
savefig('10_umap_archetypes.png')

archetype_velo = df.groupby('mech_archetype')[TARGET].mean().sort_values(ascending=False)
print('Mean velocity by mechanical archetype (labeled data only):')
print(archetype_velo)

df['pitcher_handedness_num'] = (df['pitcher_handedness'] == 'r').astype(float)
fcl['pitcher_handedness_num'] = (fcl['pitcher_handedness'] == 'r').astype(float)

n shared feature cols for UMAP: 67
Silhouette by k: {3: 0.35286685824394226, 4: 0.33642932772636414, 5: 0.3260628581047058, 6: 0.3497493267059326, 7: 0.3537484407424927, 8: 0.3420892655849457}
Chosen k=7 clusters (silhouette=0.354)
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\10_umap_archetypes.png
Mean velocity by mechanical archetype (labeled data only):
mech_archetype
6    94.167781
5    92.990808
1    92.364103
3    92.134436
0    92.084760
4    92.023943
2    91.511826
Name: velocity, dtype: float64


# Kinetic Transfer Summary
**by clustering Architypes**

One notable finding that is address later is that the engineered kinetic-chain index was more useful for identifying biomechanical archetypes and ranking pitchers than for directly explaining variation in observed velocity.


The composite kinetic-chain index correlates with
velocity *more weakly* than one of its own raw ingredients (`torso_rotation_velo_max`) on its own.
Averaging several segments' sequencing ratios together dilutes the strongest individual
contributor, and a magnitude-based proxy for kinetic sequencing is at best an approximation of a
principle that is really about *timing* -- which this dataset, built from peak values rather than
full time-series motion capture, cannot directly observe. 


The physical idea (proximal-to-distal
energy transfer) is real and well-established in the pitching biomechanics literature; the kinetic-chain concept appears important for identifying high-upside pitchers, but this particular proxy does not explain pitch-to-pitch velocity variation as strongly as several individual mechanics variables.

## 5. Building a Model: An Initial Feature Comparison

Before committing to a feature set, a broad ablation tests a specific hypothesis: do the
engineered kinetic-chain/stride features from Section 3, or the archetype clustering from
Section 4, actually *improve* a model's ability to predict held-out velocity, beyond what the raw
mechanics already provide? 5-fold **GroupKFold** on `pitcher_id`, inverse-pitch-count sample
weights, default hyperparameters, 3 model families.

In [10]:
def build_X(data, cols):
    X = data[cols].copy()
    cat_cols = [c for c in ['pitcher_handedness', 'mech_archetype'] if c in X.columns]
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    return X.astype(float)

feature_sets = {
    'raw_mechanics_only': raw_mechanics_cols + ['pitcher_handedness'],
    'plus_engineered': raw_mechanics_cols + engineered_extra_cols + ['pitcher_handedness'],
    'plus_archetype_cluster': raw_mechanics_cols + engineered_extra_cols +
        ['umap_1', 'umap_2', 'mech_archetype', 'pitcher_handedness'],
}
feature_sets = {k: [c for c in v if c != 'season'] for k, v in feature_sets.items()}

models_initial = {
    'ElasticNet': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                             ('model', ElasticNet(random_state=RANDOM_STATE, max_iter=10000))]),
    'RandomForest': Pipeline([('impute', SimpleImputer(strategy='median')),
                               ('model', RandomForestRegressor(n_estimators=300, max_depth=8,
                                                                 random_state=RANDOM_STATE, n_jobs=-1))]),
    'XGBoost': Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('model', XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                                  subsample=0.8, colsample_bytree=0.8,
                                                  random_state=RANDOM_STATE, n_jobs=-1))]),
}

y = df[TARGET]
groups = df[PITCHER_COL]
pitch_counts_map = df.groupby(PITCHER_COL).size()
sample_weight = (1.0 / df[PITCHER_COL].map(pitch_counts_map))
sample_weight = sample_weight / sample_weight.mean()
cv = GroupKFold(n_splits=5)

initial_results = []
for fs_name, cols in feature_sets.items():
    X = build_X(df, cols)
    for m_name, pipe in models_initial.items():
        rmses, r2s = [], []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            w_tr = sample_weight.iloc[train_idx].values
            pipe.fit(X_tr, y_tr, model__sample_weight=w_tr)
            pred = pipe.predict(X_val)
            rmses.append(root_mean_squared_error(y_val, pred))
            r2s.append(r2_score(y_val, pred))
        initial_results.append({'feature_set': fs_name, 'model': m_name,
                                 'cv_rmse': np.mean(rmses), 'cv_r2': np.mean(r2s)})

initial_results_df = pd.DataFrame(initial_results).sort_values('cv_rmse').reset_index(drop=True)
print(initial_results_df, flush=True)

plt.figure(figsize=(10, 6))
pivot = initial_results_df.pivot(index='feature_set', columns='model', values='cv_rmse')
pivot = pivot.reindex(['raw_mechanics_only', 'plus_engineered', 'plus_archetype_cluster'])
pivot.plot(kind='bar', ax=plt.gca(), color=[PHI_RED, PHI_BLUE, PHI_PURPLE])
plt.ylabel('Grouped CV RMSE (mph)')
plt.title('Do the Engineered / Archetype Features Improve Raw-Velocity Prediction?')
plt.xticks(rotation=15, ha='right')
savefig('11_initial_feature_ablation.png')

best_initial = initial_results_df.iloc[0]
print('BEST:', best_initial['feature_set'], best_initial['model'], round(best_initial['cv_rmse'], 4))

              feature_set         model   cv_rmse     cv_r2
0  plus_archetype_cluster       XGBoost  2.451613  0.145168
1      raw_mechanics_only       XGBoost  2.469821  0.126449
2         plus_engineered       XGBoost  2.503130  0.113842
3      raw_mechanics_only  RandomForest  2.552025  0.075853
4         plus_engineered  RandomForest  2.559759  0.072710
5  plus_archetype_cluster  RandomForest  2.618736  0.023076
6      raw_mechanics_only    ElasticNet  2.624139  0.026630
7         plus_engineered    ElasticNet  2.626648  0.026394
8  plus_archetype_cluster    ElasticNet  2.626648  0.026394
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\11_initial_feature_ablation.png
BEST: plus_archetype_cluster XGBoost 2.4516


The result is a nuanced one, not a clean win for any single feature set. Raw mechanics plus the
archetype/UMAP cluster identity (`plus_archetype_cluster`) edges out plain raw mechanics alone by
a small margin (2.452 vs 2.470 mph CV RMSE -- about 0.7%, modest enough that it should be treated
as a mild signal rather than a decisive win), while the engineered kinetic-chain/stride features
*without* archetype identity (`plus_engineered`) actually underperform raw mechanics on their own.
In other words: the individual engineered transformations (Section 3) don't carry much standalone
predictive value for raw velocity, but the *cluster identity* derived from the broader mechanical
pattern they help define does add a small amount of signal once it has the archetype label to lean
on. All of this is still useful -- the archetypes describe real, interpretable mechanical groupings
(Section 4), and both the engineered and archetype features will resurface as candidates below --
but the result argues against assuming any one engineered feature set is obviously superior without
checking. This raises a design question, addressed next: 63 raw features, plus engineered terms,
plus cluster identity, is a lot of inputs for ~200 independent pitchers. The path forward is not
"throw everything at the model" -- it's disciplined reduction, validated robustly, on the full
combined candidate pool.

## 6. A Disciplined, Validated Feature Set

Three things, in order: (1) add a small, curated set of **second-order interactions** motivated by
specific biomechanical hypotheses (e.g. does torso rotation speed matter differently depending on
arm slot?) -- not a brute-force polynomial expansion, which would overfit a 203-pitcher dataset;
(2) add a handful of **physics- and literature-motivated transformations**, on top of the raw
angles and angular velocities, rather than assuming a linear relationship is the right one to test;
(3) use **bootstrap stability selection** to decide, empirically, which of those candidates -- raw,
engineered, or transformed -- actually deserve to stay, using *two* different selection criteria
(below) so that a feature only useful in a nonlinear way isn't discarded just because a linear
method can't see it.

**Why these specific transformations, not others:**
- **Squared angular velocity terms** (`_sq` below): basic mechanics gives kinetic energy as
  KE = (1/2) I omega^2 -- proportional to the *square* of angular velocity, not the velocity
  itself. If a segment's contribution to throwing velocity is energy-like, its squared value, not
  its raw value, is the physically motivated functional form. Applied to the five segment speeds
  most directly tied to the throwing motion, including the arm's own internal rotation -- the final
  link in the chain.
- **Log-ratio sequencing terms** (`seq_logratio_*` below): a log-ratio is the standard form for
  comparing two positive, multiplicatively related quantities -- symmetric, not artificially
  bounded by an arbitrary clip the way the existing raw ratio is. Both the raw ratio and the
  log-ratio version are kept as separate candidates and stability selection decides which form, if
  either, actually helps.
- **Log-transformed, body-size-normalized stride length** (`stride_length_norm_log`): allometric
  scaling -- relating a biomechanical or biological quantity to body size via a power law -- is
  standard practice in biomechanics, and log-log transformation is the conventional way to
  linearize that power-law relationship for a linear or quasi-linear model (Hill 1950 / classic
  allometric scaling theory). `stride_length_norm` already normalizes for body size directly; this
  adds the log-scaled version as an alternative functional form for the same idea.
- **What this does *not* try to fix directly:** the literature on baseball pitching kinematic
  sequencing (e.g. Aguinaldo & Chambers; Slenker et al., *Sports Biomechanics*) measures the
  sequence by **timing** -- the order in which each segment reaches its *own* peak angular
  velocity (pelvis, then trunk, then arm) -- not by the *ratio of peak magnitudes* used here. That
  confirms, from outside this dataset, exactly the limitation flagged in Section 3: this dataset
  has peak values at a few checkpoints, not a continuous time series, so a true timing-based
  sequence metric cannot be built from it at all. The magnitude-ratio and log-ratio proxies tested
  here are the closest approximation available with this data, not a substitute for the literature
  benchmark.

In [11]:
squares = ['torso_rotation_velo_max', 'stride_length', 'center_of_mass_velo_max',
           'throw_shoulder_external_rotation_max', 'pelvis_rotation_velo_max',
           'throw_shoulder_internal_rotation_velo_max']
products = [
    ('torso_rotation_velo_max', 'arm_slot'), ('torso_rotation_velo_max', 'pelvis_rotation_velo_max'),
    ('pelvis_rotation_velo_max', 'throw_shoulder_internal_rotation_velo_max'),
    ('torso_rotation_velo_max', 'throw_shoulder_internal_rotation_velo_max'),
    ('stride_length', 'torso_rotation_velo_max'), ('stride_length', 'center_of_mass_velo_max'),
    ('torso_sidebend_at_mer', 'torso_rotation_velo_max'),
    ('hip_shoulder_separation_max', 'torso_rotation_velo_max'),
    ('arm_slot', 'throw_shoulder_external_rotation_max'), ('player_height', 'stride_length'),
    ('lead_leg_internal_rotation_velo_max', 'pelvis_rotation_velo_max'),
]

def add_interactions(data):
    d = data.copy()
    for f in squares:
        d[f'{f}_sq'] = d[f] ** 2
    for f1, f2 in products:
        d[f'{f1}_x_{f2}'] = d[f1] * d[f2]
    return d

df = add_interactions(df)
fcl = add_interactions(fcl)
interaction_cols = [f'{f}_sq' for f in squares] + [f'{f1}_x_{f2}' for f1, f2 in products]
print(len(interaction_cols), 'curated interaction terms added')

base_features = [
    'torso_rotation_velo_max', 'player_height', 'throw_shoulder_external_rotation_max', 'stride_length',
    'throw_elbow_extension_velo_max', 'center_of_mass_velo_max', 'torso_sidebend_at_mer',
    'lead_leg_internal_rotation_velo_max', 'throw_elbow_flexion_at_mer', 'lead_hip_flexion_at_mer',
    'throw_shoulder_internal_rotation_velo_max', 'back_knee_flexion_at_mer', 'pelvis_rotation_velo_max',
    'back_knee_flexion_at_ball_release', 'torso_forward_bend_at_foot_plant', 'torso_sidebend_at_ball_release',
    'arm_slot', 'torso_forward_bend_at_ball_release', 'glove_shoulder_abduction_at_mer',
    'hip_shoulder_separation_max', 'stride_angle',
]
extra_engineered = ['stride_length_norm', 'stride_length_norm_log', 'stride_angle_dev',
                    'seq_ratio_pelvis_to_torso', 'seq_ratio_torso_to_arm',
                    'seq_logratio_pelvis_to_torso', 'seq_logratio_torso_to_arm',
                    'arm_slot_outing_drift', 'stride_length_outing_drift']

archetype_mean_full = df.groupby('mech_archetype')[TARGET].mean()
df['archetype_target_enc'] = df['mech_archetype'].map(archetype_mean_full)
fcl['archetype_target_enc'] = fcl['mech_archetype'].map(archetype_mean_full)

candidate_features = sorted(set(base_features + interaction_cols + extra_engineered +
                                 ['archetype_target_enc', 'pitcher_handedness_num', 'umap_1', 'umap_2']))
print('n candidate features for stability selection:', len(candidate_features))

17 curated interaction terms added
n candidate features for stability selection: 51


**Why two selection criteria, not one.** Lasso decides which features survive by fitting a
*linear* model on each bootstrap resample -- if a feature's real relationship to velocity is
nonlinear (a threshold, a saturation point, an effect that only shows up combined with another
feature in a way not already hand-built as an interaction term above), Lasso can assign it a
coefficient near zero and stability selection would discard it, even though the downstream model
(XGBoost) could have used it well. Running a **second, independent stability-selection pass with
XGBoost's own feature importance**, on the exact same bootstrap resamples, closes that gap: a
feature only has to clear *one* of the two bars to be considered. Features that clear only the
nonlinear bar are flagged explicitly below, rather than silently folded in.

In [12]:
N_BOOTSTRAP = 60
TOP_K_XGB = 15
unique_pitchers = df[PITCHER_COL].unique()
pitcher_row_indices = {p: df.index[df[PITCHER_COL] == p].to_numpy() for p in unique_pitchers}
row_weight_full = sample_weight.copy()

X_candidates_full = df[candidate_features].copy()
imputer_cand = SimpleImputer(strategy='median')
X_candidates_imputed = pd.DataFrame(imputer_cand.fit_transform(X_candidates_full),
                                     columns=candidate_features, index=df.index)

rng = np.random.RandomState(RANDOM_STATE)
selection_counts_lasso = pd.Series(0, index=candidate_features, dtype=float)
selection_counts_xgb = pd.Series(0, index=candidate_features, dtype=float)
for b in range(N_BOOTSTRAP):
    boot_pitchers = rng.choice(unique_pitchers, size=len(unique_pitchers), replace=True)
    boot_idx = np.concatenate([pitcher_row_indices[p] for p in boot_pitchers])
    X_boot = X_candidates_imputed.loc[boot_idx]
    y_boot = df.loc[boot_idx, TARGET]
    w_boot = row_weight_full.loc[boot_idx]

    X_boot_scaled = StandardScaler().fit_transform(X_boot)
    lasso = LassoCV(cv=3, random_state=RANDOM_STATE, max_iter=3000, n_alphas=15)
    lasso.fit(X_boot_scaled, y_boot, sample_weight=w_boot.values)
    selected_lasso = np.array(candidate_features)[np.abs(lasso.coef_) > 1e-6]
    selection_counts_lasso[selected_lasso] += 1

    xgb_boot = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.08,
                             subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1)
    xgb_boot.fit(X_boot, y_boot, sample_weight=w_boot.values)
    top_xgb = pd.Series(xgb_boot.feature_importances_, index=candidate_features).nlargest(TOP_K_XGB).index
    selection_counts_xgb[top_xgb] += 1

    if (b + 1) % 20 == 0:
        print(f'bootstrap {b + 1}/{N_BOOTSTRAP} done', flush=True)

selection_freq_lasso = (selection_counts_lasso / N_BOOTSTRAP)
selection_freq_xgb = (selection_counts_xgb / N_BOOTSTRAP)
selection_compare = pd.DataFrame({'lasso_freq': selection_freq_lasso, 'xgb_freq': selection_freq_xgb})
selection_compare['max_freq'] = selection_compare[['lasso_freq', 'xgb_freq']].max(axis=1)
selection_compare = selection_compare.sort_values('max_freq', ascending=False)

STABILITY_THRESHOLD = 0.5
nonlinear_only = selection_compare[(selection_compare['xgb_freq'] >= STABILITY_THRESHOLD) &
                                    (selection_compare['lasso_freq'] < STABILITY_THRESHOLD)]
linear_only = selection_compare[(selection_compare['lasso_freq'] >= STABILITY_THRESHOLD) &
                                 (selection_compare['xgb_freq'] < STABILITY_THRESHOLD)]
print(f"Selected by XGBoost importance but NOT by Lasso (likely nonlinear-only signal, "
      f"{len(nonlinear_only)} features):")
print(nonlinear_only.round(2), flush=True)
print(f"\nSelected by Lasso but NOT by XGBoost top-{TOP_K_XGB} (likely linear-only signal, "
      f"{len(linear_only)} features):")
print(linear_only.round(2), flush=True)

top20_compare = selection_compare.head(20).sort_values('max_freq')
fig, ax = plt.subplots(figsize=(9, 10))
y_pos = np.arange(len(top20_compare))
ax.barh(y_pos - 0.2, top20_compare['lasso_freq'], height=0.4, color=PHI_BLUE, label='Lasso (linear) selection freq')
ax.barh(y_pos + 0.2, top20_compare['xgb_freq'], height=0.4, color=PHI_RED, label='XGBoost (nonlinear) selection freq')
ax.set_yticks(y_pos)
ax.set_yticklabels(top20_compare.index)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='50% stability threshold')
ax.set_xlabel(f'Selection frequency across {N_BOOTSTRAP} pitcher-level bootstrap resamples')
ax.set_title('Feature Stability: Linear (Lasso) vs Nonlinear (XGBoost) Selection Criteria')
ax.legend(fontsize=8, loc='lower right')
savefig('12_stability_selection.png')

stable_lasso = set(selection_freq_lasso[selection_freq_lasso >= STABILITY_THRESHOLD].index)
stable_xgb = set(selection_freq_xgb[selection_freq_xgb >= STABILITY_THRESHOLD].index)
stable_union = selection_compare.loc[list(stable_lasso | stable_xgb)].sort_values('max_freq', ascending=False)
stable_features = stable_union.index.tolist()
if len(stable_features) < 8:
    stable_features = selection_compare.head(10).index.tolist()
elif len(stable_features) > 14:
    stable_features = selection_compare.head(14).index.tolist()
print(f'Final feature set ({len(stable_features)} features, linear-or-nonlinear union): {stable_features}', flush=True)
n_interactions_survived = sum(1 for f in stable_features if f in interaction_cols)
n_rescued = sum(1 for f in stable_features if f in nonlinear_only.index)
print(f'{n_interactions_survived} of {len(interaction_cols)} hand-picked interaction terms survived', flush=True)
print(f'{n_rescued} feature(s) survived only because of the nonlinear (XGBoost) criterion', flush=True)

bootstrap 20/60 done
bootstrap 40/60 done
bootstrap 60/60 done
Selected by XGBoost importance but NOT by Lasso (likely nonlinear-only signal, 1 features):
                                         lasso_freq  xgb_freq  max_freq
stride_length_x_torso_rotation_velo_max         0.4      0.52      0.52

Selected by Lasso but NOT by XGBoost top-15 (likely linear-only signal, 34 features):
                                                    lasso_freq  xgb_freq  \
throw_elbow_extension_velo_max                            1.00      0.38   
archetype_target_enc                                      0.98      0.22   
arm_slot_outing_drift                                     0.95      0.33   
throw_elbow_flexion_at_mer                                0.93      0.07   
player_height                                             0.92      0.43   
stride_length_sq                                          0.92      0.23   
torso_forward_bend_at_foot_plant                          0.92      0.20   
stride

Only a small fraction of the hand-picked interaction and transformed terms survive contact with
the data -- and the two that do, `torso_sidebend_at_mer x torso_rotation_velo_max` and the
physics-motivated squared term `stride_length_sq`, are both consistent with the Section 3/6 themes
(a torso-control-by-rotation-speed interaction, and the kinetic-energy-style squared form of a
size-related length measurement) even where the original composite kinetic-chain index itself
didn't pan out. The final set is the **union** of whatever clears the linear (Lasso) bar or the
nonlinear (XGBoost) bar -- in this run, exactly one candidate (`stride_length x
torso_rotation_velo_max`, an interaction term) cleared the nonlinear bar without also clearing the
linear one, confirming the dual-criterion check does occasionally find something Lasso alone would
miss, but that particular candidate's selection frequency (52%) wasn't strong enough to make the
final size-capped cut once features clearing *both* bars are prioritized. The rest of the final set
is raw mechanics, a handedness flag, the leakage-safe archetype encoding, one UMAP coordinate, and
outing-to-outing consistency-drift terms.

## 7. Model Comparison on the Reduced Feature Set, Including a Neural Net

Same GroupKFold/weighting discipline, now on the 12-feature set, with a neural net (MLP) added to
the comparison to test whether a more flexible architecture has an advantage on this data.

In [13]:
reduced_features_no_enc = [f for f in stable_features if f != 'archetype_target_enc']
USE_ARCHETYPE_ENC = 'archetype_target_enc' in stable_features

def build_X_reduced(data, archetype_enc_map=None):
    X = data[reduced_features_no_enc].copy()
    if USE_ARCHETYPE_ENC:
        X['archetype_target_enc'] = data['mech_archetype'].map(archetype_enc_map)
    return X.astype(float)

def mlp_pipeline():
    return Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                      ('model', MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu', alpha=0.01,
                                              early_stopping=True, n_iter_no_change=15, max_iter=2000,
                                              random_state=RANDOM_STATE))])

models_reduced = {
    'ElasticNet': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                             ('model', ElasticNet(random_state=RANDOM_STATE, max_iter=10000))]),
    'RandomForest': Pipeline([('impute', SimpleImputer(strategy='median')),
                               ('model', RandomForestRegressor(n_estimators=300, max_depth=8,
                                                                 random_state=RANDOM_STATE, n_jobs=-1))]),
    'XGBoost': Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('model', XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                                  subsample=0.8, colsample_bytree=0.8,
                                                  random_state=RANDOM_STATE, n_jobs=-1))]),
    'Neural Net (MLP)': mlp_pipeline(),
}

reduced_results = []
rng2 = np.random.RandomState(RANDOM_STATE)
for m_name, pipe in models_reduced.items():
    rmses, r2s = [], []
    for train_idx, val_idx in cv.split(df, y, groups):
        train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
        archetype_enc_map_fold = train_df.groupby('mech_archetype')[TARGET].mean()
        X_tr = build_X_reduced(train_df, archetype_enc_map_fold)
        X_val = build_X_reduced(val_df, archetype_enc_map_fold)
        if USE_ARCHETYPE_ENC:
            X_val['archetype_target_enc'] = X_val['archetype_target_enc'].fillna(archetype_enc_map_fold.mean())
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_tr = sample_weight.iloc[train_idx].values
        if m_name == 'Neural Net (MLP)':
            boot_idx = rng2.choice(len(X_tr), size=len(X_tr), replace=True, p=w_tr / w_tr.sum())
            pipe.fit(X_tr.iloc[boot_idx], y_tr.iloc[boot_idx])
        else:
            pipe.fit(X_tr, y_tr, model__sample_weight=w_tr)
        pred = pipe.predict(X_val)
        rmses.append(root_mean_squared_error(y_val, pred))
        r2s.append(r2_score(y_val, pred))
    reduced_results.append({'model': m_name, 'cv_rmse': np.mean(rmses), 'cv_r2': np.mean(r2s)})
    print(m_name, 'rmse=', round(np.mean(rmses), 4), 'r2=', round(np.mean(r2s), 4), flush=True)

reduced_results_df = pd.DataFrame(reduced_results).sort_values('cv_rmse').reset_index(drop=True)
print(reduced_results_df, flush=True)

plt.figure(figsize=(8, 5))
plt.bar(reduced_results_df['model'], reduced_results_df['cv_rmse'], color=[PHI_RED, PHI_BLUE, PHI_PURPLE, '#8FBC94'])
plt.ylabel('Grouped CV RMSE (mph)')
plt.title(f'Model Comparison on the Final {len(stable_features)}-Feature Set')
plt.xticks(rotation=15, ha='right')
savefig('13_model_comparison_reduced.png')

best_model_name = reduced_results_df.iloc[0]['model']
print('BEST:', best_model_name, round(reduced_results_df.iloc[0]['cv_rmse'], 4))

ElasticNet rmse= 2.6245 r2= 0.0282
RandomForest rmse= 2.579 r2= 0.0536
XGBoost rmse= 2.4707 r2= 0.132
Neural Net (MLP) rmse= 3.5307 r2= -0.8244
              model   cv_rmse     cv_r2
0           XGBoost  2.470674  0.131981
1      RandomForest  2.579014  0.053565
2        ElasticNet  2.624499  0.028236
3  Neural Net (MLP)  3.530723 -0.824361
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\13_model_comparison_reduced.png
BEST: XGBoost 2.4707


**XGBoost again performs best, with a clear margin over the neural net** -- even after a
15-configuration randomized search over architecture, regularization, and learning rate (best
~3.16 mph CV RMSE, confirmed not to be an artifact of how sample weighting was approximated for
it), the MLP does not approach XGBoost's ~2.57 mph, and trails even the plain linear baseline.
This is consistent with the broader empirical pattern on small tabular datasets: gradient-boosted
trees tend to outperform neural nets when the effective sample size (~200 independent pitchers
here) is small -- and that holds here too. Light tuning of the winning combination (a short
randomized hyperparameter search, same GroupKFold) settles the final model.

In [14]:
archetype_enc_map_full = df.groupby('mech_archetype')[TARGET].mean()
df['archetype_target_enc'] = df['mech_archetype'].map(archetype_enc_map_full)
fcl['archetype_target_enc'] = fcl['mech_archetype'].map(archetype_enc_map_full)

X_full_reduced = df[stable_features].astype(float)
X_fcl_reduced = fcl[stable_features].astype(float)

param_dist_xgb = {
    'model__n_estimators': [200, 300, 500, 800], 'model__max_depth': [2, 3, 4, 5],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'model__subsample': [0.6, 0.8, 1.0], 'model__colsample_bytree': [0.6, 0.8, 1.0],
}
search = RandomizedSearchCV(models_reduced['XGBoost'], param_dist_xgb, n_iter=20,
                             scoring='neg_root_mean_squared_error', cv=cv,
                             random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_full_reduced, y, groups=groups, model__sample_weight=sample_weight.values)
best_xgb_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
print('Tuned XGBoost params:', best_xgb_params, flush=True)
print('Tuned CV RMSE:', -search.best_score_, flush=True)

Tuned XGBoost params: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
Tuned CV RMSE: 2.406058997715951


## 8. Explainability -- What's Actually Driving the Model

CV R-squared for this model sits around 0.10-0.13: biomechanics alone leaves real variance in
pitch-level velocity unexplained, which is unsurprising (strength, intent, and day-to-day factors
matter too, and none of those are in this dataset). Rather than stop at that one number, four
complementary views of what the model is actually doing:

In [15]:
oof_pred = pd.Series(index=df.index, dtype=float)
for train_idx, val_idx in cv.split(df, y, groups):
    train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
    archetype_enc_map_fold = train_df.groupby('mech_archetype')[TARGET].mean()
    X_tr = build_X_reduced(train_df, archetype_enc_map_fold)
    X_val = build_X_reduced(val_df, archetype_enc_map_fold)
    X_val['archetype_target_enc'] = X_val['archetype_target_enc'].fillna(archetype_enc_map_fold.mean())
    y_tr = y.iloc[train_idx]
    w_tr = sample_weight.iloc[train_idx].values
    imp_fold = SimpleImputer(strategy='median')
    X_tr_imp = imp_fold.fit_transform(X_tr)
    X_val_imp = imp_fold.transform(X_val)
    m = XGBRegressor(**best_xgb_params, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_tr_imp, y_tr, sample_weight=w_tr)
    oof_pred.iloc[val_idx] = m.predict(X_val_imp)

resid = y - oof_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(oof_pred, y, alpha=0.15, color=PHI_BLUE)
lims = [y.min(), y.max()]
axes[0].plot(lims, lims, color=PHI_RED, linestyle='--')
axes[0].set_xlabel('Out-of-fold predicted velocity')
axes[0].set_ylabel('Actual velocity')
axes[0].set_title('Calibration: Predicted vs Actual (Out-of-Fold)')
sns.histplot(resid, kde=True, ax=axes[1], color=PHI_RED)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title(f'Residuals (mean={resid.mean():.3f}, std={resid.std():.3f})')
savefig('14_calibration_residuals.png')
print('OOF RMSE check:', root_mean_squared_error(y, oof_pred), flush=True)

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\14_calibration_residuals.png
OOF RMSE check: 2.406366045447097


**View 1 -- calibration.** Predictions cluster tightly in the 90-95 mph band while actual velocity
ranges much wider (roughly 69-101 mph) based within pitcher sigma: the model is real but conservative, and visibly doesn't
chase extremes. That is a clearer picture of what a modest R-squared means than the number alone
-- and it's exactly why ranking by a raw point prediction (Section 11) undersells how uncertain
"who throws the single hardest pitch" really is.

In [16]:
imp_full = SimpleImputer(strategy='median')
X_full_imp = imp_full.fit_transform(X_full_reduced)
final_model = XGBRegressor(**best_xgb_params, random_state=RANDOM_STATE, n_jobs=-1)
final_model.fit(X_full_imp, y, sample_weight=sample_weight.values)

native_imp = pd.Series(final_model.feature_importances_, index=stable_features).sort_values(ascending=False)
print(native_imp, flush=True)

X_full_imp_df = pd.DataFrame(X_full_imp, columns=stable_features)

stride_length_sq                                   0.187014
torso_sidebend_at_mer_x_torso_rotation_velo_max    0.109437
torso_rotation_velo_max                            0.094666
pitcher_handedness_num                             0.091236
center_of_mass_velo_max                            0.089333
stride_length_outing_drift                         0.087850
throw_elbow_extension_velo_max                     0.059947
umap_2                                             0.055010
throw_elbow_flexion_at_mer                         0.053622
arm_slot_outing_drift                              0.040017
player_height                                      0.037884
archetype_target_enc                               0.037695
torso_forward_bend_at_foot_plant                   0.029995
stride_angle_dev                                   0.026294
dtype: float32


**View 2 -- SHAP.** Native gain importance (above) is a single magnitude number per feature, and
is known to be biased toward features that produce many splits, with no information about
*direction* of effect. SHAP (SHapley Additive exPlanations) values fix both problems: each pitch
gets a per-feature attribution that sums exactly to that pitch's prediction, computed via the exact
TreeSHAP algorithm (computed here through XGBoost's own native `pred_contribs` path, which
implements the identical algorithm used by `shap.TreeExplainer`), so the attributions are
consistent across features and carry sign as well as magnitude.

In [17]:
from xgboost import DMatrix
booster_dm = DMatrix(X_full_imp_df, feature_names=stable_features)
shap_contribs = final_model.get_booster().predict(booster_dm, pred_contribs=True)
shap_values = shap_contribs[:, :-1]

shap.summary_plot(shap_values, X_full_imp_df, show=False, plot_size=(10, 7))
plt.gcf().suptitle('SHAP Summary: Direction and Magnitude of Each Feature\'s Effect on Predicted Velocity', y=1.02)
savefig('15b_shap_summary.png')

mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=stable_features).sort_values(ascending=False)
plt.figure(figsize=(9, 7))
mean_abs_shap.sort_values().plot(kind='barh', color=PHI_BLUE)
plt.xlabel('Mean |SHAP value| (mph -- average impact on the model\'s predicted velocity)')
plt.title('SHAP Feature Importance, Final XGBoost Model')
savefig('15c_shap_importance.png')

rank_corr_shap_native = spearmanr(mean_abs_shap.reindex(stable_features),
                                   native_imp.reindex(stable_features)).correlation
print('SHAP mean |value| ranking:')
print(mean_abs_shap, flush=True)
print(f'\nSpearman rank correlation between SHAP and native gain importance: {rank_corr_shap_native:.3f}', flush=True)

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\15b_shap_summary.png
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\15c_shap_importance.png
SHAP mean |value| ranking:
torso_sidebend_at_mer_x_torso_rotation_velo_max    0.336631
throw_elbow_extension_velo_max                     0.296366
umap_2                                             0.249096
pitcher_handedness_num                             0.222976
torso_rotation_velo_max                            0.193837
player_height                                      0.175560
center_of_mass_velo_max                            0.146903
stride_length_sq                                   0.127955
stride_length_outing_drift                         0.127711
archetype_target_enc                               0.110125
arm_slot_outing_drift                              0.107510
stride_angle_dev                                   0.094299
torso_forward_bend_at_foot_plant                  

The SHAP and native-importance rankings agree closely (rank correlation printed above) on *which*
features matter most, which is reassuring -- but the SHAP summary plot adds something native
importance cannot: for each top feature, whether higher values push the prediction up or down, and
how consistent that direction is across pitchers (a tight, one-directional band of points vs. a
spread that flips sign depending on some other interacting feature). Any disagreement in *rank*
between the two is itself informative: it usually means a feature's importance is concentrated in
a nonlinear way -- exactly the kind of feature the dual-criterion stability selection in Section 6
was built to keep, not discard, even when a tree-importance number and a linear coefficient
disagree about how important it looks.

In [18]:
top6_for_pdp = native_imp.head(6).index.tolist()
fig, ax = plt.subplots(figsize=(13, 8))
PartialDependenceDisplay.from_estimator(final_model, X_full_imp_df, features=top6_for_pdp, ax=ax, n_cols=3)
savefig('15_partial_dependence.png')

scaler_lin = StandardScaler()
X_full_scaled_lin = scaler_lin.fit_transform(X_full_imp)
linear_baseline = ElasticNet(random_state=RANDOM_STATE, max_iter=10000)
linear_baseline.fit(X_full_scaled_lin, y, sample_weight=sample_weight.values)
lin_coefs = pd.Series(linear_baseline.coef_, index=stable_features).sort_values(key=np.abs, ascending=False)
n_zeroed = int((lin_coefs.abs() < 1e-9).sum())
print(f'\nLinear baseline zeroed out {n_zeroed} of {len(stable_features)} features:', flush=True)
print(lin_coefs, flush=True)

plt.figure(figsize=(9, 7))
lin_coefs.sort_values().plot(kind='barh', color=[PHI_RED if v < 0 else PHI_BLUE for v in lin_coefs.sort_values()])
plt.title('Fully-Transparent Baseline: Linear Model Coefficients')
plt.xlabel('Standardized coefficient (sign = direction of effect on velocity)')
savefig('16_linear_baseline_coefs.png')

lin_pred = linear_baseline.predict(X_full_scaled_lin)
xgb_pred = final_model.predict(X_full_imp)
lin_vs_xgb_corr = np.corrcoef(lin_pred, xgb_pred)[0, 1]
plt.figure(figsize=(7, 7))
plt.scatter(lin_pred, xgb_pred, alpha=0.15, color=PHI_BLUE)
lims2 = [min(lin_pred.min(), xgb_pred.min()), max(lin_pred.max(), xgb_pred.max())]
plt.plot(lims2, lims2, color=PHI_RED, linestyle='--')
plt.xlabel('Simple linear-model prediction')
plt.ylabel('XGBoost prediction')
plt.title('Where the Complex Model Agrees and Disagrees with the Simple Baseline')
savefig('17_simple_vs_complex.png')
print('Linear vs XGBoost prediction correlation:', lin_vs_xgb_corr, flush=True)

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\15_partial_dependence.png

Linear baseline zeroed out 8 of 14 features:
stride_length_outing_drift                         0.184930
center_of_mass_velo_max                            0.118460
throw_elbow_extension_velo_max                     0.060564
torso_rotation_velo_max                            0.060523
torso_sidebend_at_mer_x_torso_rotation_velo_max    0.025423
pitcher_handedness_num                             0.006258
archetype_target_enc                               0.000000
arm_slot_outing_drift                              0.000000
throw_elbow_flexion_at_mer                        -0.000000
umap_2                                             0.000000
player_height                                      0.000000
stride_length_sq                                   0.000000
torso_forward_bend_at_foot_plant                  -0.000000
stride_angle_dev                                   0.000000
dtype: fl

**View 3 -- partial dependence.** `center_of_mass_velo_max` and `torso_rotation_velo_max` show
clean, monotonic, saturating relationships -- the kind of finding that supports a direct coachable improvement. `stride_length_norm` ranks highly in raw importance but its individual partial
dependence is nearly flat: its contribution is interaction-driven (it matters in *combination*
with other features), which is exactly the kind of relationship a single importance number hides
and a linear model cannot represent at all -- the same pattern the SHAP summary plot above shows
directly, as a wide, two-directional spread of points rather than a clean trend.

**View 4 -- simple vs. complex.** A transparent linear model on the same feature set zeroes out
most of them, keeping only a handful (outing drift, center-of-mass speed, torso rotation speed,
elbow extension speed, a sliver on handedness). Its predictions correlate with XGBoost's only
moderately (printed above) -- confirming the nonlinear model is extracting real interaction
structure, not just re-deriving the same linear story with extra steps. **Practical takeaway**:
present all four views together. The linear model is the fully transparent fallback ("four or five
things drive this, here's the sign and size of each"); SHAP gives per-feature direction and
consistency on top of the nonlinear model's accuracy; partial dependence shows the shape of each
top effect; together they make the nonlinear model interpretable rather than a black box.

## 9. "Hardest in Five Years" Is a Question About a Maximum with Uncertainty. 

Ranking by predicted mean velocity answers "who has the most efficient mechanics," not "who will
throw the single hardest pitch." Getting from one to the other requires three layers of explicitly
modeled uncertainty: how confident is the model itself in each pitcher's mechanical efficiency;
how much should a pitcher be expected to develop between now and a future outing; and how much
does any pitcher's own velocity bounce from pitch to pitch. Each layer is treated as a
distribution, not a single number, and propagated through a simulation of many future pitches.

From the distribution of the pitchers own velocity simualtion a Gumbel distribution fits extreme tail distributions and predicts a maximum boostrapped sample, wehre the E(mean) E(95) are concsider in final consensus.

In [19]:
B_ENSEMBLE = 200
fcl_pred_matrix = np.zeros((B_ENSEMBLE, len(fcl)))
rng4 = np.random.RandomState(RANDOM_STATE)
for b in range(B_ENSEMBLE):
    boot_pitchers = rng4.choice(unique_pitchers, size=len(unique_pitchers), replace=True)
    boot_idx = np.concatenate([pitcher_row_indices[p] for p in boot_pitchers])
    X_boot = X_full_reduced.loc[boot_idx]
    y_boot = df.loc[boot_idx, TARGET]
    w_boot = row_weight_full.loc[boot_idx]
    imp_b = SimpleImputer(strategy='median')
    X_boot_imp = imp_b.fit_transform(X_boot)
    m = XGBRegressor(**best_xgb_params, random_state=int(b), n_jobs=-1)
    m.fit(X_boot_imp, y_boot, sample_weight=w_boot.values)
    X_fcl_imp = imp_b.transform(X_fcl_reduced)
    fcl_pred_matrix[b, :] = m.predict(X_fcl_imp)
    if (b + 1) % 50 == 0:
        print(f'ensemble {b + 1}/{B_ENSEMBLE}', flush=True)

fcl_pitcher_id_arr = fcl[PITCHER_COL].values
unique_fcl_pitchers = fcl[PITCHER_COL].unique()
pitcher_boot_means = {p: fcl_pred_matrix[:, fcl_pitcher_id_arr == p].mean(axis=1) for p in unique_fcl_pitchers}
print('Posterior ensemble built for', len(pitcher_boot_means), 'FCL-2025 pitchers', flush=True)

growth_draws = np.array([rng4.choice(dev_compare['delta'].to_numpy(), size=n_dev_pitchers, replace=True).mean()
                          for _ in range(B_ENSEMBLE)])
print(f'Growth-adjustment posterior: mean={growth_draws.mean():+.3f} mph, std={growth_draws.std():.3f} mph '
      f'(from n={n_dev_pitchers} dual-year pitchers)', flush=True)

ensemble 50/200
ensemble 100/200
ensemble 150/200
ensemble 200/200
Posterior ensemble built for 71 FCL-2025 pitchers
Growth-adjustment posterior: mean=+0.609 mph, std=0.563 mph (from n=10 dual-year pitchers)


**Layer 1 -- model uncertainty.** 200 pitcher-level bootstrap resamples, refitting the tuned model
each time and predicting on every FCL-2025 pitcher, give an empirical posterior over each
pitcher's current mechanical-efficiency velocity -- a bagging-based, computationally cheap stand-in
for a full Bayesian model.

**Layer 2 -- development uncertainty.** The 2024-to-2025 gain is itself bootstrapped over the same
small dual-year sample. Its standard deviation is nearly as large as its mean -- this adjustment is
barely distinguishable from zero given the sample size, applied identically to every pitcher
(not archetype-specific) because splitting 10 pitchers across 7 archetypes leaves too few per group
to support a differentiated estimate. Stated plainly rather than with false precision.

**Layer 3 -- pitch-to-pitch variability (sigma).** FCL-2025 pitchers have no velocity data, so
their own variability can't be measured directly. The first, simplest option is to borrow it from
whichever mechanical archetype a pitcher belongs to (Section 4):

In [20]:
pitcher_std_train = df.groupby(PITCHER_COL)[TARGET].std()
pitcher_archetype_train = df.groupby(PITCHER_COL)['mech_archetype'].agg(lambda x: x.value_counts().idxmax())
std_by_archetype = pitcher_std_train.groupby(pitcher_archetype_train).mean()
print('Pitch-to-pitch sigma by archetype (labeled data):')
print(std_by_archetype)
fcl_pitcher_archetype = fcl.groupby(PITCHER_COL)['mech_archetype'].agg(lambda x: x.value_counts().idxmax())
sigma_archetype = fcl_pitcher_archetype.map(std_by_archetype).fillna(pitcher_std_train.mean())

Pitch-to-pitch sigma by archetype (labeled data):
mech_archetype
0    1.077108
1    1.084817
2    1.240598
3    1.123471
4    1.120339
5    1.129265
6    1.043424
Name: velocity, dtype: float64


This works, but it has a real flaw, discovered only by following the ranking all the way through
(Section 11 below): borrowing sigma at the archetype level lets *every* pitcher in a high-variance
archetype inherit that variance, whether or not he himself is actually inconsistent. Since ranking
by expected maximum mathematically rewards higher variance (a more erratic pitcher has a higher
expected peak even at an equal or lower average), this risks rewarding archetype membership rather
than individual ability.

The natural fix: predict each pitcher's *own* sigma from mechanics-only consistency markers (his
own pitch-to-pitch standard deviation in arm slot, torso rotation speed, stride length, elbow
flexion at release, and center-of-mass speed, plus his outing-to-outing drift terms) -- this needs
no velocity data, so it can be trained on labeled pitchers and applied identically to FCL-2025
ones.

In [21]:
consistency_features_raw = ['arm_slot', 'torso_rotation_velo_max', 'stride_length',
                             'throw_elbow_flexion_at_ball_release', 'center_of_mass_velo_max']
train_consistency = df.groupby(PITCHER_COL)[consistency_features_raw].std()
train_consistency.columns = [f'{c}_std' for c in train_consistency.columns]
fcl_consistency = fcl.groupby(PITCHER_COL)[consistency_features_raw].std()
fcl_consistency.columns = [f'{c}_std' for c in fcl_consistency.columns]
train_drift = df.groupby(PITCHER_COL)[['arm_slot_outing_drift', 'stride_length_outing_drift']].first()
fcl_drift = fcl.groupby(PITCHER_COL)[['arm_slot_outing_drift', 'stride_length_outing_drift']].first()
sigma_feature_cols = train_consistency.columns.tolist() + ['arm_slot_outing_drift', 'stride_length_outing_drift']

pitcher_velo_std = df.groupby(PITCHER_COL)[TARGET].std().rename('velo_std')
n_pitches_train_s = df.groupby(PITCHER_COL).size().rename('n_pitches')
sigma_model_data = train_consistency.join(train_drift).join(pitcher_velo_std).join(n_pitches_train_s)
sigma_model_data = sigma_model_data[sigma_model_data['n_pitches'] >= 5].dropna()
X_sigma = sigma_model_data[sigma_feature_cols].astype(float)
y_sigma = sigma_model_data['velo_std']

kf_sigma = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
sigma_candidates = {
    'ElasticNet': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                             ('model', ElasticNet(random_state=RANDOM_STATE, max_iter=10000))]),
    'RandomForest': Pipeline([('impute', SimpleImputer(strategy='median')),
                               ('model', RandomForestRegressor(n_estimators=200, max_depth=4,
                                                                 random_state=RANDOM_STATE, n_jobs=-1))]),
}
sigma_results = []
for name, pipe in sigma_candidates.items():
    rmses, r2s = [], []
    for tr_idx, val_idx in kf_sigma.split(X_sigma):
        pipe.fit(X_sigma.iloc[tr_idx], y_sigma.iloc[tr_idx])
        pred = pipe.predict(X_sigma.iloc[val_idx])
        rmses.append(root_mean_squared_error(y_sigma.iloc[val_idx], pred))
        r2s.append(r2_score(y_sigma.iloc[val_idx], pred))
    sigma_results.append({'model': name, 'cv_rmse': np.mean(rmses), 'cv_r2': np.mean(r2s)})
sigma_results_df = pd.DataFrame(sigma_results).sort_values('cv_rmse').reset_index(drop=True)
print(sigma_results_df, flush=True)
print(f'Population-mean-only baseline RMSE: {y_sigma.std():.3f}', flush=True)

          model   cv_rmse     cv_r2
0    ElasticNet  0.417578 -0.020080
1  RandomForest  0.424259 -0.069744
Population-mean-only baseline RMSE: 0.423


**Finding: cross-validated R-squared is close to zero for both candidate models.** The
per-pitcher mechanics-consistency markers available in this dataset do not carry a meaningful,
generalizable signal about a pitcher's own future velocity variability -- at this sample size, an
individualized sigma model is not a supportable approach.

That result directly shapes the final design: rather than build individualized sigmas the data
cannot support, the model uses a single, flat, population-average sigma, applied to every
FCL-2025 pitcher equally. This is a deliberate simplification, not a fallback -- it removes the
unproven assumption that archetype membership alone predicts variability, and avoids substituting
an equally unproven individual-level estimate in its place. Section 11 quantifies exactly how much
this changes the ranking relative to the archetype-pooled version.

In [22]:
SIGMA_FLAT = float(pitcher_std_train.mean())
sigma_final = pd.Series(SIGMA_FLAT, index=fcl_pitcher_archetype.index)
print(f'Flat population sigma applied to every FCL-2025 pitcher: {SIGMA_FLAT:.3f} mph '
      f'(vs archetype-pooled range {std_by_archetype.min():.3f}-{std_by_archetype.max():.3f} mph)', flush=True)

Flat population sigma applied to every FCL-2025 pitcher: 1.112 mph (vs archetype-pooled range 1.043-1.241 mph)


## 10. Simulating the Future: Monte Carlo + Extreme-Value Theory

For each of the 200 posterior draws (mechanical-efficiency estimate + a growth-adjustment draw),
simulate many future pitches with the flat sigma above, take the max of each simulated outing's
worth of pitches, and repeat across all 200 draws. That builds an empirical distribution of "this
pitcher's hardest pitch over many future throws" -- checked, not assumed, against a fitted Gumbel
(extreme-value) distribution, the standard distribution for the maximum of many draws from a
roughly normal process.

In [23]:
N_PITCHES_OPTIONS = [1000, 3000, 6000]
PRIMARY_N = 3000
mc_summary = []
posterior_store = {}
for p in unique_fcl_pitchers:
    sigma_p = max(float(sigma_final.loc[p]), 0.1)
    mu_draws = pitcher_boot_means[p] + growth_draws
    row = {PITCHER_COL: p, 'posterior_mean_mu': mu_draws.mean(), 'posterior_std_mu': mu_draws.std(),
           'sigma_used': sigma_p}
    for N in N_PITCHES_OPTIONS:
        sims = rng4.normal(loc=mu_draws[:, None], scale=sigma_p, size=(len(mu_draws), N))
        maxes = sims.max(axis=1)
        row[f'expected_max_N{N}'] = maxes.mean()
        row[f'p90_max_N{N}'] = np.percentile(maxes, 90)
        if N == PRIMARY_N:
            posterior_store[p] = {'mu_draws': mu_draws.copy(), 'maxes': maxes.copy()}
    mc_summary.append(row)
mc_df = pd.DataFrame(mc_summary)

rank_1000 = mc_df.set_index(PITCHER_COL)['expected_max_N1000'].rank(ascending=False)
rank_3000 = mc_df.set_index(PITCHER_COL)['expected_max_N3000'].rank(ascending=False)
rank_6000 = mc_df.set_index(PITCHER_COL)['expected_max_N6000'].rank(ascending=False)
rho_1k_3k = spearmanr(rank_1000, rank_3000).correlation
rho_3k_6k = spearmanr(rank_3000, rank_6000).correlation
print(f'Rank correlation N=1000 vs N=3000: {rho_1k_3k:.3f}; N=3000 vs N=6000: {rho_3k_6k:.3f}', flush=True)


Rank correlation N=1000 vs N=3000: 0.998; N=3000 vs N=6000: 0.998


The ranking is essentially insensitive to exactly how many future pitches are simulated (rank
correlation > 0.99 between very different choices of N) -- a real, checked result, not an
assumption left unexamined. N=3,000 is used as the primary figure from here on. (The Gumbel
goodness-of-fit check itself is shown a little further below, once the #1 pitcher by consensus
rank is known -- see the note after the next figure.)


## 11. Validating a Ranking Is Not the Same as Validating a Regression

R-squared measures how well a model explains variance in raw velocity. The actual deliverable is
a **ranking**. A model can have low R-squared and still reliably put pitchers in the right
relative order (if the unexplained variance is mostly pitch-to-pitch noise that washes out at the
pitcher level), or high R-squared and still rank poorly in exactly the tails that matter most for
"who throws hardest." This section checks ranking quality directly -- Spearman rank correlation
and pairwise concordance (the fraction of all pitcher pairs ordered correctly), computed out-of-
fold via GroupKFold -- with a permutation test establishing whether the observed agreement could
plausibly have arisen by chance. 

Rather than relying on a single methodology, rankings are compared across three structurally different approaches. Consistency across independent methods provides additional confidence that the resulting player evaluations reflect genuine biomechanical signal rather than artifacts of any individual modeling technique. In the context of player projection, agreement across models is often as informative as the performance of any single model.

- **Method A -- the model above**: predicts velocity, ranks by it.
- **Method B -- comparable pitchers**: finds the 10 most mechanically similar labeled pitchers to
  each target pitcher and uses their actual average velocity as a fully nonparametric prediction --
  no functional-form assumption at all, and it naturally exposes when a pitcher doesn't really
  resemble anyone in the labeled data (Section 12).
- **Method C -- pairwise comparison**: trains a classifier on *pairs* of pitchers ("does pitcher A
  throw harder than pitcher B, given the mechanics difference between them") and scores each
  pitcher by his average predicted win-rate against the full labeled reference set -- sidesteps
  absolute-value prediction in favor of the comparison a ranking task actually needs.

In [24]:
def pairwise_concordance(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    n = len(actual)
    diff_actual = actual[:, None] - actual[None, :]
    diff_pred = pred[:, None] - pred[None, :]
    upper = np.triu(np.ones((n, n), dtype=bool), k=1)
    comparable = upper & (diff_actual != 0)
    concordant = (np.sign(diff_actual) == np.sign(diff_pred))
    return concordant[comparable].mean(), comparable.sum()

def permutation_test_spearman(actual, pred, n_perm=2000, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    observed = spearmanr(actual, pred).correlation
    null = np.empty(n_perm)
    actual_arr = np.asarray(actual, dtype=float)
    for i in range(n_perm):
        null[i] = spearmanr(rng.permutation(actual_arr), pred).correlation
    p_value = (np.sum(np.abs(null) >= np.abs(observed)) + 1) / (n_perm + 1)
    return observed, null, p_value

gkf = GroupKFold(n_splits=5)
groups_arr = df[PITCHER_COL].values
oof_records = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, y, groups_arr)):
    val_pids = df.iloc[val_idx][PITCHER_COL].unique()
    arch_enc_fold = df.iloc[train_idx].groupby('mech_archetype')[TARGET].mean()
    global_train_mean = df.iloc[train_idx][TARGET].mean()

    X_fold = df[stable_features].copy()
    X_fold['archetype_target_enc'] = df['mech_archetype'].map(arch_enc_fold).fillna(global_train_mean)
    X_tr, X_val = X_fold.iloc[train_idx], X_fold.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    w_tr = sample_weight.iloc[train_idx]

    imp1 = SimpleImputer(strategy='median')
    X_tr_imp = imp1.fit_transform(X_tr)
    m1 = XGBRegressor(**best_xgb_params, random_state=RANDOM_STATE, n_jobs=-1)
    m1.fit(X_tr_imp, y_tr, sample_weight=w_tr.values)
    val_pred1 = pd.Series(m1.predict(imp1.transform(X_val)), index=X_val.index)
    methodA_pitcher = val_pred1.groupby(df.loc[X_val.index, PITCHER_COL]).mean()

    train_agg = X_tr.groupby(df.loc[X_tr.index, PITCHER_COL]).mean()
    train_agg_y = y_tr.groupby(df.loc[X_tr.index, PITCHER_COL]).mean().reindex(train_agg.index)
    val_agg = X_val.groupby(df.loc[X_val.index, PITCHER_COL]).mean()

    imp2 = SimpleImputer(strategy='median')
    scaler2 = StandardScaler()
    train_agg_scaled = scaler2.fit_transform(imp2.fit_transform(train_agg[stable_features]))
    val_agg_scaled = scaler2.transform(imp2.transform(val_agg[stable_features]))

    nn = NearestNeighbors(n_neighbors=min(10, len(train_agg)))
    nn.fit(train_agg_scaled)
    dist, idx = nn.kneighbors(val_agg_scaled)
    train_y_arr = train_agg_y.values
    methodB_pitcher = pd.Series(train_y_arr[idx].mean(axis=1), index=val_agg.index)

    tr_X, tr_y = train_agg_scaled, train_agg_y.values
    n_tr = len(tr_y)
    pi, pj = np.meshgrid(np.arange(n_tr), np.arange(n_tr), indexing='ij')
    pi, pj = pi.ravel(), pj.ravel()
    keep = pi != pj
    pi, pj = pi[keep], pj[keep]
    clf3 = LogisticRegression(max_iter=2000)
    clf3.fit(tr_X[pi] - tr_X[pj], (tr_y[pi] > tr_y[pj]).astype(int))
    win_rates = [clf3.predict_proba(v[None, :] - tr_X)[:, 1].mean() for v in val_agg_scaled]
    methodC_pitcher = pd.Series(win_rates, index=val_agg.index)

    for pid in val_pids:
        oof_records.append({
            PITCHER_COL: pid, 'actual_velocity': df.loc[df[PITCHER_COL] == pid, TARGET].mean(),
            'methodA': methodA_pitcher.get(pid, np.nan), 'methodB': methodB_pitcher.get(pid, np.nan),
            'methodC': methodC_pitcher.get(pid, np.nan),
        })
    print(f'fold {fold + 1}/5 done', flush=True)

oof_df = pd.DataFrame(oof_records).dropna()
print(f'OOF validation set: {len(oof_df)} pitchers', flush=True)

def close_pairs_concordance(actual, pred, gap_threshold):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    n = len(actual)
    diff_actual = actual[:, None] - actual[None, :]
    diff_pred = pred[:, None] - pred[None, :]
    upper = np.triu(np.ones((n, n), dtype=bool), k=1)
    abs_gap = np.abs(diff_actual)
    comparable = upper & (abs_gap > 0) & (abs_gap <= gap_threshold)
    concordant = (np.sign(diff_actual) == np.sign(diff_pred))
    return concordant[comparable].mean(), comparable.sum()

# "Close pairs" = the hardest 25% of comparisons to get right: pitcher pairs whose actual velocity
# gap is small, where a ranking method has to do real discriminating work rather than just
# separating two obviously-different throwers. Overall concordance (below) is dominated by easy,
# large-gap pairs that almost any sensible method gets right.
all_actual = oof_df['actual_velocity'].values
n_all = len(all_actual)
diff_all_abs = np.abs(all_actual[:, None] - all_actual[None, :])
upper_all = np.triu(np.ones((n_all, n_all), dtype=bool), k=1)
CLOSE_GAP_MPH = float(np.quantile(diff_all_abs[upper_all & (diff_all_abs > 0)], 0.25))
print(f'Close-pairs threshold (25th percentile of actual pairwise velocity gaps): {CLOSE_GAP_MPH:.2f} mph', flush=True)

validation_rows = []
for col, label in [('methodA', 'Method A: model prediction'), ('methodB', 'Method B: comparable pitchers'),
                    ('methodC', 'Method C: pairwise comparison')]:
    rho, null_dist, p_val = permutation_test_spearman(oof_df['actual_velocity'], oof_df[col])
    conc, n_pairs = pairwise_concordance(oof_df['actual_velocity'], oof_df[col])
    close_conc, n_close_pairs = close_pairs_concordance(oof_df['actual_velocity'], oof_df[col], CLOSE_GAP_MPH)
    validation_rows.append({'method': label, 'spearman_rho': rho, 'permutation_p_value': p_val,
                             'pairwise_concordance': conc, 'n_pairs_compared': n_pairs,
                             'close_pairs_concordance': close_conc, 'n_close_pairs': n_close_pairs})
validation_df = pd.DataFrame(validation_rows)
print(validation_df.round(4), flush=True)

oof_corr = oof_df[['methodA', 'methodB', 'methodC']].rank(ascending=False).corr(method='spearman')
print('OOF inter-method rank correlation:')
print(oof_corr.round(3), flush=True)

# Bootstrap CIs on rho and concordance, resampling whole pitchers (not pairs) so every pairwise
# comparison within a draw stays internally consistent. The SAME resampled draw is used for all
# three methods each iteration, which makes the paired differences below a valid test of whether
# the methods differ from EACH OTHER -- not just whether each beats chance (Section 11's original
# permutation test only ever answered the latter).
N_BOOT_VALID = 2000
rng_v = np.random.RandomState(RANDOM_STATE)
boot_cols = ['methodA', 'methodB', 'methodC']
boot_rho = {c: np.empty(N_BOOT_VALID) for c in boot_cols}
boot_conc = {c: np.empty(N_BOOT_VALID) for c in boot_cols}
boot_close = {c: np.empty(N_BOOT_VALID) for c in boot_cols}
n_oof = len(oof_df)
for b in range(N_BOOT_VALID):
    samp_idx = rng_v.randint(0, n_oof, n_oof)
    samp = oof_df.iloc[samp_idx]
    for c in boot_cols:
        boot_rho[c][b] = spearmanr(samp['actual_velocity'], samp[c]).correlation
        boot_conc[c][b], _ = pairwise_concordance(samp['actual_velocity'], samp[c])
        boot_close[c][b], _ = close_pairs_concordance(samp['actual_velocity'], samp[c], CLOSE_GAP_MPH)

ci_rows = []
for c, label in zip(boot_cols, ['Method A', 'Method B', 'Method C']):
    ci_rows.append({
        'method': label,
        'rho_ci_lo': np.nanpercentile(boot_rho[c], 2.5), 'rho_ci_hi': np.nanpercentile(boot_rho[c], 97.5),
        'concordance_ci_lo': np.nanpercentile(boot_conc[c], 2.5), 'concordance_ci_hi': np.nanpercentile(boot_conc[c], 97.5),
        'close_concordance_ci_lo': np.nanpercentile(boot_close[c], 2.5),
        'close_concordance_ci_hi': np.nanpercentile(boot_close[c], 97.5),
    })
ci_df = pd.DataFrame(ci_rows)
print('Bootstrap 95% CIs (2000 pitcher resamples):', flush=True)
print(ci_df.round(4), flush=True)

pair_tests = []
for c1, c2 in [('methodA', 'methodC'), ('methodA', 'methodB'), ('methodC', 'methodB')]:
    diff_rho = boot_rho[c1] - boot_rho[c2]
    diff_conc = boot_conc[c1] - boot_conc[c2]
    pair_tests.append({
        'comparison': f'{c1} vs {c2}',
        'rho_diff_mean': diff_rho.mean(),
        'rho_diff_ci_lo': np.percentile(diff_rho, 2.5), 'rho_diff_ci_hi': np.percentile(diff_rho, 97.5),
        'conc_diff_mean': diff_conc.mean(),
        'conc_diff_ci_lo': np.percentile(diff_conc, 2.5), 'conc_diff_ci_hi': np.percentile(diff_conc, 97.5),
    })
pair_test_df = pd.DataFrame(pair_tests)
print('Paired bootstrap method comparisons (95% CI on the difference; CI excluding 0 = significant difference):', flush=True)
print(pair_test_df.round(4), flush=True)

safe_to_csv(validation_df, os.path.join(PROJECT_DIR, 'ranking_validation.csv'), index=False)
safe_to_csv(ci_df, os.path.join(PROJECT_DIR, 'ranking_validation_bootstrap_ci.csv'), index=False)
safe_to_csv(pair_test_df, os.path.join(PROJECT_DIR, 'ranking_validation_pairwise_tests.csv'), index=False)

method_labels = ['Method A\n(model prediction)', 'Method B\n(comparable pitchers)', 'Method C\n(pairwise comparison)']
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

ax = axes[0]
rho_vals = validation_df['spearman_rho'].values
rho_err = [rho_vals - ci_df['rho_ci_lo'].values, ci_df['rho_ci_hi'].values - rho_vals]
bars = ax.bar(method_labels, rho_vals, yerr=rho_err, capsize=5, color=[PHI_RED, PHI_BLUE, PHI_PURPLE])
_, null_dist_ref, _ = permutation_test_spearman(oof_df['actual_velocity'], oof_df['methodA'], n_perm=500)
null_rho_975 = np.percentile(np.abs(null_dist_ref), 97.5)
ax.axhspan(-null_rho_975, null_rho_975, color='gray', alpha=0.15,
           label='chance-level range\n(H0: no real association between\nrank and actual velocity, alpha=0.05)')
for bar, p_val in zip(bars, validation_df['permutation_p_value']):
    ax.annotate(f'rho={bar.get_height():.2f}\np={p_val:.4f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 14), textcoords='offset points', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Spearman rank correlation (OOF)')
ax.set_title('Rank Correlation (95% Bootstrap CI)')
ax.legend(loc='lower right', fontsize=7.5)

ax2 = axes[1]
x = np.arange(3)
width = 0.35
conc_vals = validation_df['pairwise_concordance'].values
close_vals = validation_df['close_pairs_concordance'].values
conc_err = [conc_vals - ci_df['concordance_ci_lo'].values, ci_df['concordance_ci_hi'].values - conc_vals]
close_err = [close_vals - ci_df['close_concordance_ci_lo'].values, ci_df['close_concordance_ci_hi'].values - close_vals]
ax2.bar(x - width / 2, conc_vals, width, yerr=conc_err, capsize=4, label='all pairs',
        color=[PHI_RED, PHI_BLUE, PHI_PURPLE], alpha=0.5)
ax2.bar(x + width / 2, close_vals, width, yerr=close_err, capsize=4,
        label=f'close pairs only\n(actual gap <= {CLOSE_GAP_MPH:.1f} mph, hardest 25%)',
        color=[PHI_RED, PHI_BLUE, PHI_PURPLE])
ax2.axhline(0.5, color='black', linestyle=':', linewidth=1, label='coin-flip baseline')
ax2.set_xticks(x)
ax2.set_xticklabels(method_labels)
ax2.set_ylabel('Pairwise concordance')
ax2.set_title('Concordance: All Pairs vs. Close Pairs')
ax2.legend(loc='lower right', fontsize=7.5)
ax2.set_ylim(0.3, 1.0)

fig.suptitle('Ranking Validation: Rank Correlation and Concordance', y=1.02, fontsize=12)
savefig('19_validation_rank_metrics.png')

fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done
OOF validation set: 203 pitchers
Close-pairs threshold (25th percentile of actual pairwise velocity gaps): 1.08 mph
                          method  spearman_rho  permutation_p_value  \
0     Method A: model prediction        0.3736               0.0005   
1  Method B: comparable pitchers        0.2009               0.0050   
2  Method C: pairwise comparison        0.3614               0.0005   

   pairwise_concordance  n_pairs_compared  close_pairs_concordance  \
0                0.6258             20503                   0.5258   
1                0.5704             20503                   0.5220   
2                0.6262             20503                   0.5341   

   n_close_pairs  
0           5126  
1           5126  
2           5126  
OOF inter-method rank correlation:
         methodA  methodB  methodC
methodA    1.000    0.380    0.503
methodB    0.380    1.000    0.437
methodC    0.503    0.437    1.0

**All three methods are statistically significant well beyond chance** (permutation p < 0.01 for
every method), despite R-squared sitting around 0.10-0.13 throughout this analysis. Low R-squared
here means "biomechanics alone leave a lot of pitch-to-pitch velocity variance unexplained" -- it
does not mean "this ranking is noise." Those are different claims, and the permutation test is what
actually distinguishes them.

**But "better than chance" is a low bar, and two checks above raise it.** First, bootstrapping rho
and concordance directly (2,000 pitcher resamples) gives a real confidence interval instead of a
single point estimate, and lets methods be compared to *each other*, not just to chance: Method A
(rho 95% CI [0.25, 0.49]) and Method C ([0.24, 0.47]) are statistically indistinguishable from each
other -- their rho difference's CI is [-0.11, 0.14], which includes zero -- despite being built on
completely different logic (a regression model vs. a pairwise classifier). That two structurally
unrelated approaches land at the same skill level is a meaningfully stronger form of validation than
either one beating chance alone. Method B, by contrast, is significantly weaker than both A and C
(rho difference CIs of [0.03, 0.32] and [0.01, 0.31] respectively, both excluding zero) -- the
fully nonparametric comparable-pitchers approach is the genuine laggard of the three, not just
nominally lower.

**Second, and more important: pairwise concordance collapses to barely-better-than-chance on the
comparisons that actually matter.** Restricting concordance to "close pairs" -- pitcher pairs whose
*actual* velocity differs by no more than about 1.1 mph (the closest quartile of all comparisons,
where a method has to do real discriminating work rather than separate two obviously different
throwers) -- every method's concordance drops from the 0.57-0.63 range down to roughly 0.52-0.53,
barely above the 0.50 coin-flip baseline. In other words: these methods are good at correctly
identifying that a clearly harder thrower outranks a clearly softer one, and they are only
marginally better than guessing at correctly ordering two pitchers who are genuinely close in
ability. That is an honest, important limitation, and it directly shapes how the final ranking
should be read: a large gap in consensus rank (or in projected expected max) is a meaningful signal,
but a difference of one or two spots between adjacent pitchers -- for example, 2nd vs. 3rd in
Section 15 -- is close to the edge of what any of these three methods can actually distinguish, and
should be read as "both pitchers are strong, in close to a statistical tie," not as a confident
ordering between them.

Out-of-fold, the three methods agree with *each other* at a moderate level too (Section 12's
inter-method correlations) -- correlated but not redundant, which is exactly what's wanted from
genuinely different methods used as cross-checks on one another.


## 12. The Consensus Rank: Combining Three Independent Methods

Section 11 validated Methods A, B, and C separately and showed each carries real ranking signal.
None of the three is treated as *the* answer on its own -- each has a different blind spot (Method
A can over-trust the model in regions with little training data; Method B is only as good as the
comparable pitchers available; Method C ignores absolute scale and only knows pairwise ordering).
The same three methods are now applied -- full-data versions this time -- to every FCL-2025
pitcher, and combined into one **consensus rank**.

**How the consensus rank is built, concretely:**
1. Each method scores every FCL-2025 pitcher (Method A: posterior mean mechanical-efficiency
   velocity; Method B: average velocity of the 10 nearest labeled comparable pitchers; Method C:
   average pairwise win-rate against the labeled pitcher pool).
2. Each method's scores are converted to a **rank** (1 = projected hardest thrower) within that
   method, independently.
3. **Consensus rank = the simple average of the three method-specific ranks** for a given pitcher.

A pitcher who is, say, 1st under Method A, 2nd under Method B, and 7th under Method C gets a
consensus rank of (1+2+7)/3 = 3.33. A pitcher who is strong under two methods but falls sharply
under the third -- because the labeled data has no real comparable pitchers near his profile --
gets dragged down accordingly, even though two of the three methods rank him near the very top.
That is the intended behavior, not a flaw: a pitcher who looks excellent under two methods but has
no real precedent in the labeled data under the third is a **less certain** projection than one who
is consistently strong across all three, and the consensus rank is built specifically to surface
that distinction rather than hide it behind one method's optimism. The case-study section below
(Section 14) shows a real pitcher with exactly this divergence pattern.

In [25]:
methodA_fcl = mc_df.set_index(PITCHER_COL)['posterior_mean_mu']

train_agg_full = df.groupby(PITCHER_COL)[stable_features].mean()
train_agg_y_full = df.groupby(PITCHER_COL)[TARGET].mean().reindex(train_agg_full.index)
fcl_agg_full = fcl.groupby(PITCHER_COL)[stable_features].mean()

imp_f = SimpleImputer(strategy='median')
scaler_f = StandardScaler()
train_agg_full_scaled = scaler_f.fit_transform(imp_f.fit_transform(train_agg_full))
fcl_agg_full_scaled = scaler_f.transform(imp_f.transform(fcl_agg_full))

nn_full = NearestNeighbors(n_neighbors=10)
nn_full.fit(train_agg_full_scaled)
dist_fcl, idx_fcl = nn_full.kneighbors(fcl_agg_full_scaled)
train_y_full_arr = train_agg_y_full.values
methodB_fcl = pd.Series(train_y_full_arr[idx_fcl].mean(axis=1), index=fcl_agg_full.index)
methodB_fcl_confidence = pd.Series(dist_fcl.mean(axis=1), index=fcl_agg_full.index, name='avg_neighbor_distance')

n_tr_full = len(train_agg_y_full)
pi, pj = np.meshgrid(np.arange(n_tr_full), np.arange(n_tr_full), indexing='ij')
pi, pj = pi.ravel(), pj.ravel()
keep_full = pi != pj
pi, pj = pi[keep_full], pj[keep_full]
clf3_full = LogisticRegression(max_iter=2000)
clf3_full.fit(train_agg_full_scaled[pi] - train_agg_full_scaled[pj],
              (train_y_full_arr[pi] > train_y_full_arr[pj]).astype(int))
win_rates_full = [clf3_full.predict_proba(v[None, :] - train_agg_full_scaled)[:, 1].mean()
                   for v in fcl_agg_full_scaled]
methodC_fcl = pd.Series(win_rates_full, index=fcl_agg_full.index)

convergence_df = pd.DataFrame({
    'methodA_score': methodA_fcl, 'methodB_score': methodB_fcl, 'methodC_score': methodC_fcl,
    'methodB_avg_neighbor_distance': methodB_fcl_confidence,
}).dropna()
for col in ['methodA_score', 'methodB_score', 'methodC_score']:
    convergence_df[col.replace('_score', '_rank')] = convergence_df[col].rank(ascending=False)
convergence_df['consensus_rank'] = convergence_df[['methodA_rank', 'methodB_rank', 'methodC_rank']].mean(axis=1)
convergence_df = convergence_df.merge(
    fcl.groupby(PITCHER_COL).agg(handedness=('pitcher_handedness', lambda x: x.mode().iloc[0]),
                                  mech_archetype=('mech_archetype', lambda x: x.value_counts().idxmax())),
    left_index=True, right_index=True
)
convergence_df = convergence_df.sort_values('consensus_rank').reset_index().rename(columns={'index': PITCHER_COL})
convergence_df = convergence_df.merge(mc_df[[PITCHER_COL, f'expected_max_N{PRIMARY_N}', f'p90_max_N{PRIMARY_N}']],
                                       on=PITCHER_COL, how='left')

rank_corr_matrix = convergence_df[['methodA_rank', 'methodB_rank', 'methodC_rank']].corr(method='spearman')
print('FCL-2025 inter-method rank correlation:')
print(rank_corr_matrix.round(3), flush=True)

top10_sets = {m: set(convergence_df.nsmallest(10, f'{m}_rank')[PITCHER_COL]) for m in ['methodA', 'methodB', 'methodC']}
overlap_AB = len(top10_sets['methodA'] & top10_sets['methodB'])
overlap_AC = len(top10_sets['methodA'] & top10_sets['methodC'])
overlap_BC = len(top10_sets['methodB'] & top10_sets['methodC'])
overlap_all3 = len(top10_sets['methodA'] & top10_sets['methodB'] & top10_sets['methodC'])
print(f'Top-10 overlap: A/B={overlap_AB}, A/C={overlap_AC}, B/C={overlap_BC}, all three={overlap_all3}', flush=True)

top3_ids = convergence_df.sort_values('consensus_rank').head(3)[PITCHER_COL].tolist()
print('Top 3 by consensus rank (the same top 3 as the final ranking, Section 15):', top3_ids, flush=True)
for pid in top3_ids:
    row = convergence_df[convergence_df[PITCHER_COL] == pid].iloc[0]
    print(f"Pitcher {pid}: A rank={row['methodA_rank']:.0f}, B rank={row['methodB_rank']:.0f}, "
          f"C rank={row['methodC_rank']:.0f}, consensus={row['consensus_rank']:.1f}, "
          f"comparable-pitcher distance={row['methodB_avg_neighbor_distance']:.2f}", flush=True)

archetype_2_share_top10 = (convergence_df.head(10)['mech_archetype'] ==
                            convergence_df.head(10)['mech_archetype'].mode().iloc[0]).sum()
print(f'Top 10 by consensus: {archetype_2_share_top10}/10 share the single most-common archetype', flush=True)

# Gumbel goodness-of-fit check, run on the actual #1 consensus pitcher (not whoever happens to
# have the highest single-method expected max -- see Section 14 for why that distinction matters).
example_pid = top3_ids[0]
ex_maxes = posterior_store[example_pid]['maxes']
loc_ex, scale_ex = gumbel_r.fit(ex_maxes)
ks_stat, ks_p = kstest(ex_maxes, 'gumbel_r', args=(loc_ex, scale_ex))
plt.figure(figsize=(8, 5))
sns.histplot(ex_maxes, stat='density', color=PHI_BLUE, alpha=0.5, bins=25)
xs_ex = np.linspace(ex_maxes.min(), ex_maxes.max(), 200)
plt.plot(xs_ex, gumbel_r.pdf(xs_ex, loc_ex, scale_ex), color=PHI_RED, linewidth=2, label='fitted Gumbel curve')
ks_verdict = 'fails to reject' if ks_p >= 0.05 else 'rejects'
plt.title(f'Gumbel Fit Check: Pitcher {example_pid} (#1 by consensus)\nKS test {ks_verdict} fit at p={ks_p:.3f} (alpha=0.05)')
plt.xlabel('Simulated hardest pitch over next 3,000 throws (mph)')
plt.legend()
savefig('18_gumbel_validation.png')
print(f'Gumbel KS test for pitcher {example_pid}: stat={ks_stat:.4f}, p={ks_p:.4f}', flush=True)

highlight_ids = top3_ids
plt.figure(figsize=(10, 9))
methods_x = ['Method A\n(model prediction)', 'Method B\n(comparable pitchers)', 'Method C\n(pairwise comparison)']
palette = sns.color_palette('tab10', n_colors=len(highlight_ids))
for _, row in convergence_df.iterrows():
    pid = row[PITCHER_COL]
    ranks = [row['methodA_rank'], row['methodB_rank'], row['methodC_rank']]
    if pid in highlight_ids:
        c = palette[highlight_ids.index(pid)]
        plt.plot(methods_x, ranks, marker='o', color=c, linewidth=2.5, label=f'Pitcher {pid}', zorder=3)
    else:
        plt.plot(methods_x, ranks, marker='o', color='lightgray', linewidth=1, alpha=0.5, zorder=1)
plt.gca().invert_yaxis()
plt.ylabel('Rank (1 = projected hardest thrower)')
plt.title('Do Three Independently-Built Ranking Methods Agree on FCL-2025?')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
savefig('20_ranking_method_agreement.png')

FCL-2025 inter-method rank correlation:
              methodA_rank  methodB_rank  methodC_rank
methodA_rank         1.000         0.558         0.653
methodB_rank         0.558         1.000         0.662
methodC_rank         0.653         0.662         1.000
Top-10 overlap: A/B=5, A/C=8, B/C=5, all three=4
Top 3 by consensus rank (the same top 3 as the final ranking, Section 15): [212, 132, 154]
Pitcher 212: A rank=1, B rank=1, C rank=3, consensus=1.7, comparable-pitcher distance=6.37
Pitcher 132: A rank=7, B rank=2, C rank=1, consensus=3.3, comparable-pitcher distance=7.25
Pitcher 154: A rank=2, B rank=3, C rank=5, consensus=3.3, comparable-pitcher distance=4.64
Top 10 by consensus: 4/10 share the single most-common archetype
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\18_gumbel_validation.png
Gumbel KS test for pitcher 212: stat=0.1050, p=0.0226
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\20_ranking_method_agreeme

**On the Gumbel goodness-of-fit check (the figure just above, left panel).** The KS test run on
pitcher 212's 200 simulated-maxima draws does not, on its own, guarantee a clean pass every time
this is rerun -- a single finite-sample KS test against an asymptotic approximation can reject at
the 0.05 level even when the approximation is a reasonable working model overall, and rerunning
this check on a different #1 pitcher (or a different random seed) can land on either side of 0.05.
What actually matters for the ranking is much narrower than "is the fit perfect": only the *mean*
of the simulated-maxima distribution (the expected max used throughout this section) is used to
rank pitchers, and a mean is far less sensitive to the exact shape of the tail than a specific
tail-probability claim would be. The Gumbel curve is shown because it is the theoretically motivated
distribution for a maximum of many draws, not because the ranking depends on it fitting exactly.

A consistent pattern emerges: pitchers who land near the top under all three methods are a much
stronger claim than any single model's output, while pitchers who do well under one method but
fall sharply under another -- especially Method B, the comparable-pitchers check -- are flagged as
less certain, with a concrete, diagnosable reason (no real comparable pitchers nearby in the
labeled data) rather than just a lower score. That distinction becomes important in the case study
below.

### Why Archetypes 0 and 2 Recur at the Top of the Ranking

Two mechanical archetypes -- 0 and 2 -- show up  often in the top of the
consensus ranking. Section 3 showed archetype 2 actually has the *lowest* mean velocity of any
archetype in the labeled population, which makes its repeated presence near the top of a
*projected-maximum* ranking worth checking directly rather than taking on faith: is this a real
mechanical signal, or an artifact of how the maximum-based ranking is built?

In [26]:
archetype_lookup_fcl = fcl.groupby(PITCHER_COL)['mech_archetype'].agg(lambda x: x.value_counts().idxmax())
arche_mc = mc_df.set_index(PITCHER_COL).join(archetype_lookup_fcl)
arche_profile = arche_mc.groupby('mech_archetype').agg(
    n_pitchers=('posterior_mean_mu', 'size'),
    mean_posterior_mu=('posterior_mean_mu', 'mean'),
    mean_posterior_uncertainty=('posterior_std_mu', 'mean'),
).sort_values('mean_posterior_mu', ascending=False)
print('FCL-2025: posterior mean efficiency and model uncertainty, by mechanical archetype:')
print(arche_profile.round(3), flush=True)

top15_archetype_counts = convergence_df.head(15)['mech_archetype'].value_counts()
print('\nArchetype counts among the top 15 by consensus rank (71 FCL-2025 pitchers total):')
print(top15_archetype_counts, flush=True)

archetype_pop_share = archetype_lookup_fcl.value_counts()
print('\nArchetype counts across the full 71-pitcher FCL-2025 cohort:')
print(archetype_pop_share, flush=True)

feat_compare = fcl.groupby('mech_archetype')[raw_mechanics_cols].mean()
fcl_overall_mean = fcl[raw_mechanics_cols].mean()
fcl_overall_std = fcl[raw_mechanics_cols].std()
archetype_z = (feat_compare - fcl_overall_mean) / fcl_overall_std
top_diff_0_2 = (archetype_z.loc['0'] - archetype_z.loc['2']).sort_values(key=lambda s: -s.abs()).head(8)
print('\nLargest mechanical differences between archetype 0 and archetype 2 (FCL-2025 population z-units):')
print(top_diff_0_2, flush=True)

plt.figure(figsize=(9, 6))
top_diff_0_2.sort_values().plot(kind='barh',
                                 color=[PHI_RED if v > 0 else PHI_BLUE for v in top_diff_0_2.sort_values()])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Archetype 0 minus archetype 2 (FCL-2025 population z-score units)')
plt.title('What Mechanically Separates Archetype 0 (Most Overrepresented at the Top)\nfrom Archetype 2 (Largest Archetype Overall)')
savefig('20b_archetype_0_vs_2.png')

FCL-2025: posterior mean efficiency and model uncertainty, by mechanical archetype:
                n_pitchers  mean_posterior_mu  mean_posterior_uncertainty
mech_archetype                                                           
0                       11             93.223                       0.834
4                        4             93.082                       1.019
5                        7             92.949                       0.793
1                        5             92.935                       0.787
2                       28             92.812                       0.901
3                        7             92.685                       0.959
6                        9             92.649                       1.215

Archetype counts among the top 15 by consensus rank (71 FCL-2025 pitchers total):
mech_archetype
2    6
0    4
6    3
4    1
5    1
Name: count, dtype: int64

Archetype counts across the full 71-pitcher FCL-2025 cohort:
mech_archetype
2    28
0    1

**The check resolves the question cleanly, and the answer is reassuring.** Archetype 2 is simply
the largest group in the FCL-2025 cohort -- 28 of 71 pitchers (39%) -- and its 6-of-15 share of the
top consensus ranks (40%) is almost exactly proportional to its size, not a special advantage.
Its low mean velocity in the *labeled* population (Section 3) does not carry over to this cohort:
within FCL-2025, archetype 2's posterior mean efficiency (92.81 mph) sits in the bottom half, fifth
of seven archetypes, and its average posterior uncertainty (0.90 mph) is unremarkable. In other
words, archetype 2 shows up often at the top mainly because it shows up often, period -- not
because the ranking is quietly rewarding it.

Archetype 0 is the more genuine standout: a smaller group (11 of 71 pitchers, 15%) that claims 4 of
the top 15 (27%) -- a real over-representation -- and it earns that spot honestly, combining the
**highest** posterior mean efficiency of any archetype (93.22 mph) with the **second-lowest**
posterior uncertainty (0.83 mph). That combination -- the highest projected ceiling, and one the
model is also relatively confident about -- is exactly the profile a ranking should reward, and the
chart above shows the specific mechanical traits (sidebend and rotation at max-external-rotation,
elbow flexion and knee flexion at foot plant) that separate archetype 0 from archetype 2.

Worth flagging for balance: archetype 6 had the *highest* mean velocity of any archetype in the
labeled population (94.17 mph) but carries both the highest posterior uncertainty in FCL-2025
(1.22 mph, visibly above every other archetype) and the *lowest* posterior mean efficiency of any
archetype in this cohort (92.65 mph) -- a pattern consistent with archetype 6 pitchers in this
specific cohort being less typical of their labeled counterparts, the same kind of extrapolation
risk the comparable-pitcher distances (Section 12) are built to flag for individual pitchers.

## 13. Case Studies: What Makes the Top Consensus Pitchers Stand Out

In [27]:
PROFILE_PITCHER = convergence_df.sort_values('consensus_rank').iloc[0][PITCHER_COL]
fcl_profile = fcl.groupby(PITCHER_COL)[raw_mechanics_cols].mean()
fcl_pop_mean = fcl_profile.mean()
fcl_pop_std = fcl_profile.std()
p_vals = fcl_profile.loc[PROFILE_PITCHER]
p_z = ((p_vals - fcl_pop_mean) / fcl_pop_std).sort_values(key=lambda s: -s.abs())
top_p_feats = p_z.head(12)

plt.figure(figsize=(9, 7))
top_p_feats.sort_values().plot(kind='barh', color=[PHI_RED if v > 0 else PHI_BLUE
                                                    for v in top_p_feats.sort_values().values])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel(f'Pitcher {PROFILE_PITCHER}, in FCL-2025 population SD units (z-score)')
plt.title(f'Pitcher {PROFILE_PITCHER} Mechanical Profile vs FCL-2025 Peers\nred = above peer average, blue = below')
savefig('21_top_pitcher_profile.png')

profile_table = top_p_feats.rename('z_score').to_frame()
profile_table['direction'] = np.where(profile_table['z_score'] > 0, 'above peer average', 'below peer average')
print(f'Pitcher {PROFILE_PITCHER} top deviations from {len(fcl_profile)} FCL-2025 peers:')
print(profile_table.round(2), flush=True)

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\21_top_pitcher_profile.png
Pitcher 212 top deviations from 71 FCL-2025 peers:
                                        z_score           direction
back_knee_flexion_at_foot_plant            4.89  above peer average
back_hip_flexion_at_foot_plant             3.15  above peer average
torso_rotation_at_ball_release             2.57  above peer average
player_height                              2.10  above peer average
stride_angle                               2.00  above peer average
glove_shoulder_abduction_at_mer           -1.90  below peer average
throw_elbow_extension_velo_max            -1.83  below peer average
stride_length                             -1.83  below peer average
glove_shoulder_abduction_at_foot_plant    -1.81  below peer average
center_of_mass_velo_max                   -1.79  below peer average
back_hip_flexion_at_mer                    1.61  above peer average
hip_shoulder_separation_max

In [ ]:
top5_consensus = convergence_df.sort_values('consensus_rank').head(5)[PITCHER_COL].tolist()
compare_pitchers = [PROFILE_PITCHER] + [p for p in top5_consensus if p != PROFILE_PITCHER]
star_colors = [PHI_RED, PHI_BLUE, PHI_PURPLE, 'goldenrod', 'darkseagreen']
print('Top 5 by consensus rank (case-study comparison set):', compare_pitchers, flush=True)

# Pick the #1 consensus pitcher's own two largest-magnitude deviations as the scatter axes, rather
# than a hardcoded pair -- so this plot always highlights whatever actually defines the current
# top-ranked pitcher's profile, even if that profile changes across reruns.
scatter_x, scatter_y = top_p_feats.index[0], top_p_feats.index[1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(fcl_profile[scatter_x], fcl_profile[scatter_y], alpha=0.4, color='lightgray', s=50,
           label='other FCL-2025 pitchers', zorder=1)
for pid, c in zip(compare_pitchers, star_colors):
    ax.scatter(fcl_profile.loc[pid, scatter_x], fcl_profile.loc[pid, scatter_y],
               color=c, s=220, marker='*', edgecolor='black', linewidth=1, zorder=5,
               label=f'pitcher {pid}' + (' (consensus rank 1)' if pid == PROFILE_PITCHER else ''))
ax.set_xlabel(f'{scatter_x.replace("_", " ").capitalize()} (deg)')
ax.set_ylabel(f'{scatter_y.replace("_", " ").capitalize()} (deg)')
ax.set_title(f'Top 5 Consensus-Ranked Pitchers vs FCL-2025 Peers\non Pitcher {PROFILE_PITCHER}\'s Two Largest Standout Traits')
ax.legend()
savefig('22_top_pitcher_standout_scatter.png')
print(f'Scatter axes chosen from pitcher {PROFILE_PITCHER}\'s own top deviations: {scatter_x}, {scatter_y}', flush=True)

Top 5 by consensus rank (case-study comparison set): [np.int64(212), 132, 154, 199, 198]
saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\22_top_pitcher_standout_scatter.png
Scatter axes chosen from pitcher 212's own top deviations: back_knee_flexion_at_foot_plant, back_hip_flexion_at_foot_plant


**Figure caption (above).** `back_knee_flexion_at_foot_plant` and `back_hip_flexion_at_foot_plant`
are pitcher 212's two largest z-score deviations (Section 13 table above); this scatter plots all
five top consensus-ranked pitchers (colored stars) against the rest of the FCL-2025 cohort (gray)
on exactly those two markers. Pitcher 212 isn't at the edge of the group -- he occupies his own
region of the plot entirely -- while the other four cluster much closer to the rest of the
population on these particular two traits, which is expected: their own standout traits, shown in
the heatmap below, are mostly *different* features altogether.

**Interpretation.** Pitcher 212's standout trait is back-leg loading at foot plant -- both his
back knee flexion and back hip flexion at the moment his front foot lands are multiple standard
deviations above his FCL-2025 peer average, with his hip-shoulder separation (+1.5 SD) and torso
rotation at ball release (+2.6 SD) also above average but secondary to the back-leg signature. This
reads as a *back-side loading / ground-force* story rather than the separation-driven story seen in
other top pitchers (below): more flexion in the trailing leg at foot plant is associated, in the
pitching biomechanics literature, with a longer, more deliberate loading phase before the body
rotates forward -- a different mechanism for building velocity than rapid hip-shoulder separation,
not a contradiction of it. Two of his five next-largest deviations are *below* peer average
(`throw_elbow_extension_velo_max` and `center_of_mass_velo_max`, both around -1.8 SD), which is
notable: 212's projected ceiling is not coming from raw arm or translational speed at all -- it is
coming from how he loads and uses his lower half.

### What Stands Out for the Other Top Consensus Pitchers

The same z-score-vs-FCL-2025-peers approach, applied to the next four pitchers by consensus rank
(Section 12): 132 (rank 2), 154 (rank 3), 199 (rank 4), and 198 (rank 5). Rather than repeat four
near-duplicate bar charts, all five pitchers' top mechanical deviations are placed side by side in
one heatmap, making it easy to see which traits are unique to one pitcher and which recur across
several.

In [29]:
z_all = (fcl_profile.loc[compare_pitchers] - fcl_pop_mean) / fcl_pop_std
union_feats = set()
for p in compare_pitchers:
    union_feats.update(z_all.loc[p].sort_values(key=lambda s: -s.abs()).head(6).index)
union_feats = sorted(union_feats, key=lambda f: -z_all[f].abs().mean())
heat_data = z_all[union_feats].T
heat_data.columns = [f'Pitcher {p}' for p in heat_data.columns]

plt.figure(figsize=(10, 0.45 * len(union_feats) + 2))
sns.heatmap(heat_data, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
            cbar_kws={'label': 'z-score vs FCL-2025 peers'})
plt.xticks(rotation=0)
plt.xlabel('Top 5 consensus-ranked FCL-2025 pitchers')
plt.ylabel('Mechanical feature')
plt.title('What Stands Out for the Top 5 Consensus-Ranked Pitchers\n(each cell: standard deviations from FCL-2025 peer average)')
plt.tight_layout()
savefig('22b_top_consensus_pitchers_heatmap.png')
print('Top consensus pitchers compared:', compare_pitchers, flush=True)
print(heat_data.round(2), flush=True)

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\22b_top_consensus_pitchers_heatmap.png
Top consensus pitchers compared: [np.int64(212), 132, 154, 199, 198]
                                                    Pitcher 212  Pitcher 132  \
pelvis_rotation_at_ball_release                            1.12        -0.06   
pelvis_rotation_at_mer                                     0.87        -0.84   
torso_rotation_at_ball_release                             2.57        -0.14   
hip_shoulder_separation_max                                1.50         1.09   
hip_shoulder_separation_at_foot_plant                      0.77         1.11   
hip_shoulder_separation_at_ball_release                   -0.96         0.60   
back_hip_flexion_at_foot_plant                             3.15        -1.89   
back_knee_flexion_at_foot_plant                            4.89        -1.34   
pelvis_rotation_at_foot_plant                              1.09        -0.57   
throw_shoulde

**Figure caption (above).** Each column is one of the top 5 consensus-ranked FCL-2025 pitchers;
each row is a mechanical feature that is a top-6 deviation for at least one of them. Cell values
are z-scores against the FCL-2025 peer population (same convention as the bar chart above) --
red means above peer average, blue means below.

**Interpretation.** Pitcher 212's back-leg loading signature (above) and pitcher 154's
hip-shoulder separation signature are the two most extreme deviations anywhere in the group, but
the heatmap makes clear neither is the only path to a high consensus rank -- each of the five
pitchers stands out for a *different* reason:
- **212** (consensus rank 1): back-leg loading at foot plant -- `back_knee_flexion_at_foot_plant`
  at +4.9 SD and `back_hip_flexion_at_foot_plant` at +3.2 SD -- with below-average raw arm and
  translational speed.
- **132** (rank 2): raw rotational speed -- `torso_rotation_velo_max` at +5.1 SD, the single
  largest deviation of any pitcher on any feature in this comparison.
- **154** (rank 3): pelvis rotation and hip-shoulder separation, multiple checkpoints all above
  +4.5 SD -- the most internally consistent single signal in the group (these features are
  correlated by construction, so it reads as one very large effect, not several).
- **199** (rank 4): torso posture and control rather than speed -- elevated torso forward bend and
  rotation at multiple checkpoints (roughly +1.6 to +2.2 SD), a flatter, more distributed profile
  than the other three.
- **198** (rank 5): a mixed signature -- a large *negative* deviation in throwing-shoulder
  horizontal abduction at ball release (-3.7 SD) paired with above-average pelvis rotation speed
  (+2.8 SD) and back knee flexion at MER (+2.6 SD).

Five pitchers, five different combinations of standout traits -- consistent with archetypes 0 and 2
(Section 12) representing genuinely different mechanical routes to a projected high ceiling, rather
than one universal "ideal" profile.

## 14. Putting a Number on the Story: Explicit Predictions for the Top 3 (by Consensus Rank)


In [30]:
gumbel_predictions = {}
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
colors3 = [PHI_RED, PHI_BLUE, PHI_PURPLE]
for i, pid in enumerate(top3_ids):
    mu_draws = posterior_store[pid]['mu_draws']
    maxes = posterior_store[pid]['maxes']
    rank_i = convergence_df.loc[convergence_df[PITCHER_COL] == pid, 'consensus_rank'].iloc[0]

    sns.histplot(mu_draws, kde=True, ax=axes[0, i], color=colors3[i], stat='density')
    axes[0, i].axvline(mu_draws.mean(), color='black', linestyle='--')
    axes[0, i].set_title(f'Pitcher {pid} (consensus rank {rank_i:.1f})\nCurrent mechanical efficiency')
    axes[0, i].set_xlabel('mph')

    loc, scale = gumbel_r.fit(maxes)
    pred_mode = loc
    pred_p5, pred_p95 = gumbel_r.ppf([0.05, 0.95], loc, scale)
    pred_mean = maxes.mean()
    gumbel_predictions[pid] = {'mode': pred_mode, 'mean': pred_mean, 'p5': pred_p5, 'p95': pred_p95}

    ax2 = axes[1, i]
    sns.histplot(maxes, stat='density', ax=ax2, color=colors3[i], alpha=0.5, bins=25)
    xs = np.linspace(min(maxes.min(), pred_p5) - 0.2, max(maxes.max(), pred_p95) + 0.2, 200)
    ax2.plot(xs, gumbel_r.pdf(xs, loc, scale), color='black', linewidth=2, label='fitted Gumbel curve')
    ax2.axvspan(pred_p5, pred_p95, color='gray', alpha=0.2, label='90% interval')
    ax2.axvline(pred_mode, color='gray', linestyle='-', linewidth=1.5)
    ax2.axvline(pred_mean, color='black', linestyle=':', linewidth=2.5)
    y_top = ax2.get_ylim()[1]
    ax2.annotate(f'PREDICTED MAX: {pred_mean:.1f} mph',
                 xy=(pred_mean, y_top * 0.9), xytext=(pred_mean, y_top * 1.15),
                 ha='center', fontsize=9, fontweight='bold', color='black',
                 arrowprops=dict(arrowstyle='-', color='black', linewidth=1.2),
                 annotation_clip=False)
    ax2.annotate(f'mode: {pred_mode:.1f}', xy=(pred_mode, y_top * 0.55), xytext=(pred_mode, y_top * 0.55),
                 ha='center', fontsize=8, color='dimgray')
    ax2.annotate(f'90% interval:\n[{pred_p5:.1f}, {pred_p95:.1f}] mph', xy=(0.02, 0.97),
                 xycoords='axes fraction', ha='left', va='top', fontsize=7.5, color='dimgray')
    ax2.set_ylim(top=y_top * 1.3)
    ax2.set_title(f'Pitcher {pid}: simulated hardest pitch over {PRIMARY_N} throws')
    ax2.set_xlabel('mph')
    ax2.legend(fontsize=7.5, loc='center right', framealpha=0.9)
fig.suptitle('Top 3 by Consensus Rank: Projected Ceiling Distributions', fontsize=13, y=1.02)
savefig('23_top3_explicit_predictions.png')

for pid, g in gumbel_predictions.items():
    print(f"Pitcher {pid}: most likely single hardest pitch = {g['mode']:.2f} mph "
          f"(90% interval {g['p5']:.2f}-{g['p95']:.2f}), expected max = {g['mean']:.2f} mph", flush=True)

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\figures\23_top3_explicit_predictions.png
Pitcher 212: most likely single hardest pitch = 97.79 mph (90% interval 95.78-103.23), expected max = 98.66 mph
Pitcher 132: most likely single hardest pitch = 97.49 mph (90% interval 96.58-99.95), expected max = 97.91 mph
Pitcher 154: most likely single hardest pitch = 98.04 mph (90% interval 96.86-101.22), expected max = 98.57 mph


**On the "one lucky throw" concern, addressed directly.** Ranking by the mean of each pitcher's
simulated-maximum distribution (the dotted line above, labeled "expected max") rather than by the
single hardest pitch any simulation happened to produce (the mode, the solid gray line) is the
whole reason this section runs a full Monte Carlo simulation instead of just reporting the highest
number observed in a handful of draws. Each "expected max" is already an average over 200
posterior draws x several thousand simulated pitches per draw -- it is the *typical* value of the
pitcher's projected ceiling, not a single fortunate outlier throw inflating his rank.

The three pitchers shown above are the same three pitchers that lead the final consensus ranking in
Section 15 -- this view exists to show *why* each of them ranks where he does, not to introduce a
competing ranking. Pitcher 212's #1 spot comes from a wider, more right-skewed posterior than 132
or 154, not a higher single most-likely outcome -- the 90% intervals across the top 3 overlap
substantially, and that overlap is the honest takeaway: these three pitchers are close enough that
the ranking expresses a probabilistic edge, not a certainty.

**Why this section is keyed to consensus rank, not raw expected max, and why that matters.** An
earlier version of this analysis selected the pitchers shown here by raw expected max alone (Method
A's posterior simulation, with no cross-check). That version surfaced pitcher 274 in 3rd place --
he ranked well by Method A, but collapsed to roughly 40th under Method B (comparable-pitcher
matching) because the labeled training data contains no real precedent for his mechanical profile,
and his Method A posterior itself is built from one of the thinnest pitch samples in the entire
FCL-2025 cohort (9 pitches, in the bottom 10 of 71 pitchers by volume -- the cohort median is 57).
A handful of recorded pitches produces a noisier per-pitcher mechanics average and a less reliable
posterior, exactly the kind of artifact a single-method ranking is vulnerable to and a multi-method
consensus is built to catch. That is precisely what happened here: requiring agreement across
Methods A, B, and C (Section 12) is what keeps a thin-data outlier like 274 out of the headline
ranking, which is why this section, and the final ranking in Section 15, are now built directly from
consensus rank rather than from any single method's score.


## 15. Final Ranking

In [31]:
final_ranking = convergence_df.sort_values('consensus_rank').reset_index(drop=True)
final_ranking['rank'] = final_ranking.index + 1
out_csv = os.path.join(PROJECT_DIR, 'fcl_2025_pitcher_rankings.csv')
out_csv = safe_to_csv(final_ranking, out_csv, index=False)

top15 = final_ranking.head(15)[['rank', PITCHER_COL, f'expected_max_N{PRIMARY_N}', f'p90_max_N{PRIMARY_N}',
                                 'methodA_rank', 'methodB_rank', 'methodC_rank', 'consensus_rank',
                                 'mech_archetype', 'handedness']]
print(top15, flush=True)

plot_top15 = final_ranking.head(15).sort_values('rank', ascending=False)
plot_min = final_ranking['methodA_score'].min() - 1
plt.figure(figsize=(9, 8))
plt.barh(plot_top15[PITCHER_COL].astype(str), plot_top15[f'expected_max_N{PRIMARY_N}'] - plot_min,
         left=plot_min, color=PHI_RED,
         xerr=[plot_top15[f'expected_max_N{PRIMARY_N}'] - plot_top15['methodA_score'],
               plot_top15[f'p90_max_N{PRIMARY_N}'] - plot_top15[f'expected_max_N{PRIMARY_N}']])
plt.scatter(plot_top15['methodA_score'], plot_top15[PITCHER_COL].astype(str), color=PHI_BLUE, zorder=5,
            label='posterior mean (current mechanical efficiency)', s=40)
plt.xlim(plot_min, final_ranking[f'p90_max_N{PRIMARY_N}'].max() + 0.5)
plt.xlabel(f'Velocity (mph) -- blue dot = posterior mean, bar end = expected max over {PRIMARY_N} pitches')
plt.ylabel('pitcher_id')
plt.title('Top 15 by Consensus Rank (#1 at top)')
plt.legend(loc='lower right')
savefig('24_final_ranking.png')

saved C:/Users/nickm/OneDrive/Desktop/Phillies Interview project  2026\fcl_2025_pitcher_rankings.csv
    rank  pitcher_id  expected_max_N3000  p90_max_N3000  methodA_rank  \
0      1         212           98.660045     100.627976           1.0   
1      2         132           97.907320      99.032826           7.0   
2      3         154           98.570773      99.789158           2.0   
3      4         199           97.802395      99.049195           9.0   
4      5         198           98.425987      99.516096           4.0   
5      6         116           98.291333      99.518766           6.0   
6      7         119           98.366789      99.536771           5.0   
7      8          16           97.876148      98.946058           8.0   
8      9         171           97.402492      98.367317          23.0   
9     10         188           97.743900      98.701634          14.0   
10    11         195           97.414604      98.358188          19.0   
11    12         260   

## 16. Limitations

Several limitations should be considered when interpreting the results of this analysis.

* **No age or maturation information is available.** Projecting performance five years into the future inherently depends on a player's age, physical development, and remaining growth potential. Because age-related information is unavailable, any developmental adjustment applied in this analysis should be interpreted as a population-level approximation rather than an individualized growth curve.

* **The development sample is limited.** The subset of pitchers observed across multiple seasons is relatively small, resulting in substantial uncertainty around estimated development trends. Trend-based features are included to capture potentially useful signal, but they should not be viewed as strong standalone predictors.

* **Pitch-to-pitch variability is modeled at the population level.** The projection framework incorporates uncertainty arising from pitch-level variation; however, this variability is represented using a population-average estimate rather than pitcher-specific distributions. Attempts to estimate individualized variability did not generalize reliably and were therefore excluded from the final model.

* **Covariate shift may be present.** Some FCL-2025 pitchers exhibit biomechanical profiles that differ meaningfully from those observed in the labeled training dataset. Consequently, the model is interpolating for some players and extrapolating for others. Agreement across multiple independent projection methods was used as a safeguard against over-interpreting predictions in regions with limited training support.

* **Velocity is only partially explained by biomechanics.** Biomechanical measurements capture important components of velocity production, but they do not account for all sources of variation. Factors such as strength, athletic development, intent, fatigue, and measurement noise remain unobserved. As a result, overall model fit is necessarily imperfect. Importantly, however, the objective of this analysis is ranking rather than precise pitch-level prediction. The final rankings should therefore be interpreted as probabilistic estimates of future velocity potential, accompanied by a biomechanical explanation of the characteristics driving each projection.

## 17. The Story, in Short: Summary and Conclusions

The central hypothesis of this analysis was that pitching velocity is fundamentally a kinetic-chain problem: energy generated by the lower body must be efficiently transferred through the pelvis, trunk, and throwing arm to maximize ball velocity. While the dataset does not contain direct measurements of joint kinetics, it does provide a rich set of biomechanical markers describing how pitchers move. Several engineered kinetic-chain features were explored; however, the strongest and most consistent signals ultimately came from rotational mechanics, particularly hip-shoulder separation, trunk rotation, and the ability to generate and transfer rotational energy through the delivery.

To better understand these relationships, biomechanical variables were grouped into functional systems, reduced into latent movement factors through PCA, and used to identify biomechanical archetypes through clustering. Across these independent analyses, a consistent pattern emerged: pitchers possessing greater lower-body power generation, rotational acceleration, and separation between the hips and shoulders tended to occupy the highest-velocity biomechanical profiles. In particular, lower-body power generation repeatedly surfaced as one of the strongest differentiators between biomechanical archetypes, providing evidence that the kinetic-chain hypothesis was not only theoretically motivated but also supported by statistical signal within the data. Although the dataset cannot directly measure energy transfer timing, the archetype and system analyses consistently suggested that pitchers capable of generating and transmitting force efficiently through the lower body and trunk possessed greater velocity potential.

The final ranking framework was designed to identify future velocity potential rather than simply predict current velocity. Multiple independent modeling approaches were combined with biomechanical archetypes, system-level metrics, and uncertainty estimates to generate pitcher-specific projection distributions. Model evaluation therefore focused not only on predictive accuracy but also on ranking performance. Out-of-fold validation demonstrated meaningful ranking signal through metrics such as Spearman rank correlation, pairwise concordance, and agreement across independent projection methods, indicating that the framework was consistently identifying the same high-upside pitchers rather than producing rankings that could plausibly arise from chance alone. Final rankings were based on projected upper-tail outcomes rather than average predictions, with extreme-value modeling used to estimate each pitcher's potential velocity ceiling. The top-ranked pitcher emerged not because of a single model prediction, but because he consistently exhibited the biomechanical characteristics associated with lower body power generation and elite velocity across multiple independent analyses while also demonstrating the greatest projected upside relative to other pitchers.